# moonshot_compounding — Stages 4–6 (canonical pipeline)

Point-in-time daily feature panel for the NSE Nifty 500 universe, plus a separate monthly Amihud file. This is the clean, reproducible version: every path routed through one resolver, idempotent, and **day-independent** (finds the newest version of each file regardless of the date in its name).

## What this notebook does

| Stage | Builds | Output |
|---|---|---|
| **4** | Daily PIT membership flag | `daily_pit_flag_v1_*.parquet` |
| **3B** | Remediated raw price panel (masks phantom rows, split splices, bad prints) | `raw_price_panel_consolidated_v2_*.parquet` |
| **5a** | Returns (overnight, intraday, daily) | `returns_panel_v2_*.parquet` |
| **5b** | Turnover + realized volatility (60/120) | `features_daily_v2_*.parquet` |
| **5c** | Monthly Amihud illiquidity | `amihud_monthly_v2_*.parquet` |
| **6** | Join PIT flag as metadata + manifest | `feature_panel_with_pit_v2_*.parquet`, `manifest.yaml` |

## How to run

Run top to bottom. Each **build** cell is followed by a **diagnostic** cell that gates the next stage — if a diagnostic FAILs, stop and investigate before continuing.

**Idempotency:** build cells refuse to overwrite a file already built *today*. To re-read finished outputs without rebuilding, just run the diagnostic cells — they resolve the newest file automatically. To force a rebuild, delete the target file first (never overwritten silently — that's the convention that protects validated work).

## Inputs required (built in earlier stages, read-only here)

- `raw/prices/raw_price_panel_consolidated_v1_*.parquet`
- `raw/prices/raw_price_panel_consolidated_v2_*.parquet` *(if 3B already run; else this notebook builds it)*
- `universe/canonical_membership_semiannual.parquet`
- `universe/pull_ticker_list_v1.csv`
- `corporate_actions/batches/*.parquet` *(split events, for the 3B splice check)*

## Key ratified decisions (baked in — see manifest for full list)

- **Q3** flag is tri-state (1/0/NaN); NaN before 2010-06-30, never fabricated as 0.
- **Q6** Amihud is a separate monthly file, never broadcast onto daily rows (would leak).
- **Q8** realized vol uses full window (min_periods=window), ddof=1.
- **Remediation principle:** when a price is unknown, say so (NaN). Never substitute a plausible number.

## Cell 0 — Mount Google Drive

Run once per session. If it errors with *"mountpoint must not already contain files"*, the mountpoint has stale local folders — run the cleanup in the next cell's comment, then retry.

In [1]:
# =============================================================================
# CELL 0 — MOUNT DRIVE
# =============================================================================
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError as e:
    # Stale/blocked mountpoint: clear the LOCAL scaffolding (never your Drive)
    # and retry. /content/drive is a local attach point inside the Colab VM;
    # removing it cannot touch files stored on Google Drive itself.
    print(f"Mount raised: {e}")
    print("Clearing the local mountpoint and retrying (this does NOT delete "
          "anything on your Google Drive)...")
    import shutil
    shutil.rmtree('/content/drive', ignore_errors=True)
    drive.mount('/content/drive')

print("\
Mounted. Run Cell A next.")

Mounted at /content/drive
Mounted. Run Cell A next.


## Cell A — Config + Resolver  *(run every session, after mount)*

Centralises every file path. `resolve(key)` finds the newest existing file for reads; `out_path(key)` gives a dated path for writes and refuses same-day overwrites. The preflight catches a stale mount immediately with a clear message instead of a cryptic error later.

In [10]:
# =============================================================================
# CELL A — CONFIG + RESOLVER
# One place resolves every file. Day-independent: reads find the NEWEST file
# regardless of the date in its name. Preflight catches a stale mount up front.
# =============================================================================
from pathlib import Path
from datetime import datetime
import pandas as pd

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

BASE = Path('/content/drive/MyDrive/moonshot_compounding')
BUILD_DATE = datetime.now().strftime('%Y-%m-%d')

def _preflight():
    if not Path('/content/drive').exists():
        raise RuntimeError(
            "Google Drive is NOT mounted. Run Cell 0 first. If it IS mounted "
            "but this fires, do Runtime > Disconnect and delete runtime, then "
            "remount — the mount is stale.")
    if not BASE.exists():
        raise RuntimeError(
            f"{BASE} does not exist on the mounted Drive. Either the wrong "
            "account is mounted or the mount is stale. Verify at "
            "drive.google.com, then remount cleanly.")
    n = len(list(BASE.rglob('*.parquet')))
    if n == 0:
        raise RuntimeError(
            f"{BASE} exists but contains ZERO parquet files — almost always a "
            "STALE MOUNT. Fix: Runtime > Disconnect and delete runtime, "
            "reconnect, remount. Your data is on Drive; this session can't see "
            "it. Confirm the files are present at drive.google.com first.")
    print(f"  preflight OK — {n} parquet file(s) visible under project")

REGISTRY = {
    # inputs (read-only here)
    'price_panel_v1':      dict(folder='raw/prices', pattern='raw_price_panel_consolidated_v1_*.parquet', stem=None),
    'membership':          dict(folder='universe',   pattern='canonical_membership_semiannual.parquet',   stem=None),
    'ticker_map':          dict(folder='universe',   pattern='pull_ticker_list_v1.csv',                    stem=None),
    # stage 3B output (raw v2 — SUPERSEDED by v3, retained)
    'price_panel_v2':      dict(folder='raw/prices', pattern='raw_price_panel_consolidated_v2_*.parquet', stem='raw_price_panel_consolidated_v2'),
    # stage 3B output (raw v3 — CURRENT raw panel; v2 + market-closed dates removed)
    'price_panel_v3':      dict(folder='raw/prices', pattern='raw_price_panel_consolidated_v3_*.parquet', stem='raw_price_panel_consolidated_v3'),
    # stage 4-6 outputs
    'daily_pit_flag':      dict(folder='universe',   pattern='daily_pit_flag_v1_*.parquet',                stem='daily_pit_flag_v1'),
    'returns_panel_v2':    dict(folder='features',   pattern='returns_panel_v2_*.parquet',                 stem='returns_panel_v2'),
    'features_daily_v2':   dict(folder='features',   pattern='features_daily_v2_*.parquet',                stem='features_daily_v2'),
    'amihud_monthly_v2':   dict(folder='features',   pattern='amihud_monthly_v2_*.parquet',                stem='amihud_monthly_v2'),
    'feature_panel_with_pit_v2': dict(folder='features', pattern='feature_panel_with_pit_v2_*.parquet',    stem='feature_panel_with_pit_v2'),
    'manifest':            dict(folder='.',          pattern='manifest.yaml',                              stem='manifest'),
}

def resolve(key, required=True):
    """Newest existing file for `key`, or None. Use for READS."""
    spec = REGISTRY[key]
    folder = BASE if spec['folder'] == '.' else BASE / spec['folder']
    hits = sorted(folder.glob(spec['pattern']))
    if not hits:
        if required:
            raise FileNotFoundError(
                f"No file for '{key}' matching '{spec['pattern']}' in {folder}. "
                "Either the upstream cell has not run, or the mount is stale. "
                "Re-run Cell A to distinguish (the preflight checks the mount).")
        return None
    return hits[-1]

def out_path(key, overwrite=False):
    """Dated path for a NEW output. Refuses a same-day overwrite."""
    spec = REGISTRY[key]
    if spec['stem'] is None:
        raise ValueError(f"'{key}' is an input, not an output. Use resolve().")
    folder = BASE if spec['folder'] == '.' else BASE / spec['folder']
    folder.mkdir(parents=True, exist_ok=True)
    target = (folder / 'manifest.yaml') if key == 'manifest' \
             else folder / f"{spec['stem']}_{BUILD_DATE}.parquet"
    if target.exists() and not overwrite:
        raise FileExistsError(
            f"{target.name} already exists (built today). This does not "
            f"overwrite. To RE-READ it, use resolve('{key}') in the diagnostic. "
            "To rebuild, delete the file or pass overwrite=True deliberately.")
    if key != 'manifest':
        older = [p for p in sorted(folder.glob(spec['pattern'])) if p != target]
        if older:
            print(f"  NOTE: {len(older)} older-dated '{key}' file(s) exist; reads use "
                  f"the newest, which will be {target.name} after this build.")
    return target

print("=" * 78)
print("CONFIG + RESOLVER")
print("=" * 78)
print(f"  BASE       : {BASE}")
print(f"  BUILD_DATE : {BUILD_DATE}")
_preflight()
print("\
  Artifact status:")
for k in REGISTRY:
    p = resolve(k, required=False)
    print(f"    {k:<28}: {p.name if p else '(not yet built)'}")
print("\
  resolve('<key>') to READ, out_path('<key>') to WRITE.")

# -----------------------------------------------------------------------------
# RAW PANEL POINTER + MEASURED ROW COUNT
#
# Every downstream stage reads the raw panel through RAW_PANEL_KEY and asserts
# against EXPECTED_ROWS. Both are set ONCE here.
#
# WHY THIS EXISTS: the row-count asserts used to hardcode 3,332,879 in seven
# places. When Stage 3B started deleting market-closed dates, every one of them
# became stale simultaneously. EXPECTED_ROWS is MEASURED from the panel on
# disk, so the assert tests the real invariant -- "this stage preserved the row
# count" -- rather than a literal that has to be maintained by hand.
#
# The assert is NOT weakened by this. It still halts on any unintended row-count
# change; it just no longer needs editing when the raw panel legitimately changes.
# -----------------------------------------------------------------------------
RAW_PANEL_KEY = 'price_panel_v3'

_raw_now = resolve(RAW_PANEL_KEY, required=False)
if _raw_now is None:
    EXPECTED_ROWS = None
    print(f"\
  RAW_PANEL_KEY = '{RAW_PANEL_KEY}'  -> NOT YET BUILT.")
    print("  Run Stage 3B (build) to create it. EXPECTED_ROWS will be set on the")
    print("  next run of Cell A, and downstream row-count asserts are SKIPPED")
    print("  until then (they print a loud warning rather than passing silently).")
else:
    EXPECTED_ROWS = len(pd.read_parquet(_raw_now, columns=['ticker']))
    print(f"\
  RAW_PANEL_KEY = '{RAW_PANEL_KEY}'  -> {_raw_now.name}")
    print(f"  EXPECTED_ROWS = {EXPECTED_ROWS:,} (measured from the raw panel on disk)")

def assert_rows(n, label):
    """Assert a frame's row count equals the raw panel's.

    Halts if EXPECTED_ROWS is known and does not match. If EXPECTED_ROWS is
    None (raw panel not yet built), prints a loud SKIP rather than silently
    passing -- a check that quietly disappears is worse than one that fails.
    """
    if EXPECTED_ROWS is None:
        print(f"  [SKIP] {label} row-count assert -- EXPECTED_ROWS unset "
              f"(raw panel not built; re-run Cell A after Stage 3B)")
        return
    assert n == EXPECTED_ROWS, (
        f"{label}: row count {n:,} != raw panel {EXPECTED_ROWS:,}. STOP.")

print("=" * 78)

CONFIG + RESOLVER
  BASE       : /content/drive/MyDrive/moonshot_compounding
  BUILD_DATE : 2026-07-24
  preflight OK — 1589 parquet file(s) visible under project
  Artifact status:
    price_panel_v1              : raw_price_panel_consolidated_v1_2026-07-15.parquet
    membership                  : canonical_membership_semiannual.parquet
    ticker_map                  : pull_ticker_list_v1.csv
    price_panel_v2              : raw_price_panel_consolidated_v2_2026-07-22.parquet
    price_panel_v3              : raw_price_panel_consolidated_v3_2026-07-24.parquet
    daily_pit_flag              : daily_pit_flag_v1_2026-07-24.parquet
    returns_panel_v2            : returns_panel_v2_2026-07-24.parquet
    features_daily_v2           : features_daily_v2_2026-07-22.parquet
    amihud_monthly_v2           : amihud_monthly_v2_2026-07-22.parquet
    feature_panel_with_pit_v2   : feature_panel_with_pit_v2_2026-07-22.parquet
    manifest                    : manifest.yaml
  resolve('<key>') to

## Stage 4 — Daily PIT membership flag  (build)

Expands 33 semi-annual NIFTY 500 snapshots into a daily `(Date, ticker, in_nifty500_flag)` panel via an explicit `as_of` join. Membership is **metadata** — it never filters features. Tri-state flag: `1`/`0`/`NaN` (NaN before the first snapshot, per Q3).

In [3]:
# =============================================================================
# STAGE 4 — DAILY PIT FLAG (build)   [requires Cell A]
# =============================================================================
import numpy as np

PRICE_PANEL = resolve('price_panel_v1')
MEMBERSHIP  = resolve('membership')
TICKER_MAP  = resolve('ticker_map')
OUT_FLAG    = out_path('daily_pit_flag')
OUT_NOPRICE = BASE / 'validation' / f'pit_constituents_without_prices_{BUILD_DATE}.csv'
(BASE / 'validation').mkdir(parents=True, exist_ok=True)

FIRST_SNAPSHOT_HOLD = pd.Timestamp('2010-06-30')
LAST_SNAPSHOT_HOLD  = pd.Timestamp('2026-06-01')

print("=" * 78); print("STAGE 4 — DAILY PIT FLAG (build)"); print("=" * 78)

# --- 1. Load ---
px = pd.read_parquet(PRICE_PANEL, columns=['Date', 'ticker', 'source'])
px['Date'] = pd.to_datetime(px['Date'])
print(f"\
[1] price panel : {len(px):,} rows, {px['ticker'].nunique():,} tickers")
assert px['ticker'].nunique() == 1069, "Expected 1,069 tickers. STOP."
mem = pd.read_parquet(MEMBERSHIP); mem['snap_date'] = pd.to_datetime(mem['snap_date'])
tmap = pd.read_csv(TICKER_MAP)
print(f"    membership   : {len(mem):,} rows, {mem['snap_date'].nunique()} snapshots")
print(f"    ticker map   : {len(tmap):,} rows")

# --- 2. Verify assumptions ---
print("\
[2] VERIFYING ASSUMPTIONS")
snap_dates = np.sort(mem['snap_date'].unique())
assert len(snap_dates) == 33, f"Expected 33 snapshots, found {len(snap_dates)}"
assert pd.Timestamp(snap_dates[0]) == FIRST_SNAPSHOT_HOLD, "First snapshot moved. STOP."
assert pd.Timestamp(snap_dates[-1]) == LAST_SNAPSHOT_HOLD, "Last snapshot moved. STOP."
mem_valid = mem[mem['exch_symbol'].notna() & (mem['exch_symbol'] != '#N/A')]
n_multi = (mem_valid.groupby('isin')['exch_symbol'].nunique() > 1).sum()
assert n_multi == 0, "ISIN->symbol not 1:1. STOP."
assert 'isin' in tmap.columns and tmap['isin'].duplicated().sum() == 0, "Ticker map bad. STOP."
print("    33 snapshots, boundaries confirmed, ISIN 1:1, tmap clean")

# --- 3. Resolve ticker column ---
cands = [c for c in tmap.columns if c.lower() in ('ticker','yf_ticker','symbol_ns','yahoo_ticker')]
assert len(cands) == 1, f"Ambiguous ticker column: {cands}. STOP."
TCOL = cands[0]; print(f"\
[3] ticker column: '{TCOL}'")

# --- 4. Map ISIN -> panel ticker ---
mem_mapped = mem.merge(tmap[['isin', TCOL]].rename(columns={TCOL: 'ticker'}),
                       on='isin', how='left', validate='many_to_one')
unmapped = mem_mapped[mem_mapped['ticker'].isna()]
print(f"\
[4] mapped {mem_mapped['ticker'].notna().sum():,}/{len(mem_mapped):,}; "
      f"unmapped ISINs: {unmapped['isin'].nunique()} (expected 5)")
if len(unmapped):
    (unmapped.groupby(['isin','name']).agg(first_snap=('snap_date','min'),
        last_snap=('snap_date','max'), n_snapshots=('snap_date','nunique'))
        .reset_index().assign(reason='excluded_from_pull_ticker_list_no_price_data')
        .to_csv(OUT_NOPRICE, index=False))
    print(f"    -> logged {OUT_NOPRICE.name}")
panel_tickers = set(px['ticker'].unique())
mem_mapped = mem_mapped[mem_mapped['ticker'].notna() & mem_mapped['ticker'].isin(panel_tickers)].copy()

# --- 5. Snapshot membership sets ---
snap_members = mem_mapped.groupby('snap_date')['ticker'].apply(lambda s: set(s.unique()))
snap_source = mem[['snap_date','snapshot_source']].drop_duplicates().set_index('snap_date')['snapshot_source']
assert not snap_source.index.duplicated().any(), "snap_date has >1 source. STOP."

# --- 6. Explicit as_of ---
print("\
[6] EXPANDING TO DAILY VIA EXPLICIT as_of JOIN")
def resolve_as_of(as_of_date, snapshot_dates):
    prior = snapshot_dates[snapshot_dates <= as_of_date]
    return None if len(prior) == 0 else prior[-1]
trading_days = np.sort(px['Date'].unique())
day_to_snap = {d: resolve_as_of(pd.Timestamp(d), snap_dates) for d in trading_days}
n_pre = sum(1 for v in day_to_snap.values() if v is None)
print(f"    {len(trading_days):,} trading days; {n_pre} before first snapshot (flag=NaN)")

# --- 7. Emit ---
print("\
[7] EMITTING")
flag = px[['Date', 'ticker']].copy()
flag['effective_snap_date'] = flag['Date'].map(day_to_snap)
flag['in_nifty500_flag'] = np.nan
for sd in snap_dates:
    members = snap_members.get(pd.Timestamp(sd), set())
    m = flag['effective_snap_date'] == sd
    flag.loc[m, 'in_nifty500_flag'] = flag.loc[m, 'ticker'].isin(members).astype(float)
flag.loc[flag['effective_snap_date'].isna(), 'in_nifty500_flag'] = np.nan
flag['flag_is_extrapolated'] = flag['Date'] > LAST_SNAPSHOT_HOLD
flag['snapshot_source'] = flag['effective_snap_date'].map(snap_source)
flag = flag.sort_values(['ticker', 'Date']).reset_index(drop=True)

assert len(flag) == len(px), "Flag row count != price panel. STOP."
nan_after = flag[flag['in_nifty500_flag'].isna() & (flag['Date'] >= FIRST_SNAPSHOT_HOLD)]
assert len(nan_after) == 0, "NaN flags after first snapshot. STOP."
print(f"    flag==1: {(flag['in_nifty500_flag']==1).sum():,}  "
      f"flag==0: {(flag['in_nifty500_flag']==0).sum():,}  "
      f"NaN: {flag['in_nifty500_flag'].isna().sum():,}  "
      f"extrapolated: {flag['flag_is_extrapolated'].sum():,}")

flag.to_parquet(OUT_FLAG, index=False)
print(f"\
  -> written: {OUT_FLAG.name}  ({len(flag):,} rows)")
print("=" * 78); print("STAGE 4 BUILD COMPLETE — run the diagnostic."); print("=" * 78)

  NOTE: 5 older-dated 'daily_pit_flag' file(s) exist; reads use the newest, which will be daily_pit_flag_v1_2026-07-24.parquet after this build.
STAGE 4 — DAILY PIT FLAG (build)
[1] price panel : 3,332,879 rows, 1,069 tickers
    membership   : 16,507 rows, 33 snapshots
    ticker map   : 1,071 rows
[2] VERIFYING ASSUMPTIONS
    33 snapshots, boundaries confirmed, ISIN 1:1, tmap clean
[3] ticker column: 'yf_ticker'
[4] mapped 16,502/16,507; unmapped ISINs: 5 (expected 5)
    -> logged pit_constituents_without_prices_2026-07-24.csv
[6] EXPANDING TO DAILY VIA EXPLICIT as_of JOIN
    4,243 trading days; 127 before first snapshot (flag=NaN)
[7] EMITTING
    flag==1: 1,975,928  flag==0: 1,271,487  NaN: 85,464  extrapolated: 26,977
  -> written: daily_pit_flag_v1_2026-07-24.parquet  (3,332,879 rows)
STAGE 4 BUILD COMPLETE — run the diagnostic.


### Stage 4 — diagnostic *(gate)*

Verifies integrity, the tri-state boundary, the extrapolation boundary, flagged-vs-true counts (prediction: shortfall ≤ 5, only in recent snapshots), real index changes (HDFC/RANBAXY), transition clustering, and the 2018 carry-forward tag. Uses the corrected type-safe transition check.

In [4]:
# =============================================================================
# STAGE 4 — DIAGNOSTIC (gate)   READ-ONLY
# =============================================================================
import numpy as np

flag = pd.read_parquet(resolve('daily_pit_flag')); flag['Date'] = pd.to_datetime(flag['Date'])
px = pd.read_parquet(resolve('price_panel_v1'), columns=['Date','ticker']); px['Date'] = pd.to_datetime(px['Date'])
mem = pd.read_parquet(resolve('membership')); mem['snap_date'] = pd.to_datetime(mem['snap_date'])

print("=" * 78); print("STAGE 4 DIAGNOSTIC"); print("=" * 78)
FAILURES = []
def check(name, cond, detail=""):
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}" + (f"  — {detail}" if detail else ""))
    if not cond: FAILURES.append(name)

print("\
[A] INTEGRITY")
check("row count matches price panel", len(flag) == len(px), f"{len(flag):,} vs {len(px):,}")
check("ticker count == 1,069", flag['ticker'].nunique() == 1069)
check("no duplicate (Date,ticker)", not flag.duplicated(['Date','ticker']).any())

print("\
[B] TRI-STATE BOUNDARY (NaN strictly before 2010-06-30)")
FIRST = pd.Timestamp('2010-06-30')
nan_rows = flag[flag['in_nifty500_flag'].isna()]
check("NaN flags exist", len(nan_rows) > 0)
check("all NaN strictly before first snapshot", (nan_rows['Date'] < FIRST).all() if len(nan_rows) else True)
check("no NaN on/after first snapshot", len(flag[flag['in_nifty500_flag'].isna() & (flag['Date'] >= FIRST)]) == 0)

print("\
[C] EXTRAPOLATION BOUNDARY (tagged strictly after 2026-06-01)")
LAST = pd.Timestamp('2026-06-01')
ext = flag[flag['flag_is_extrapolated']]
check("extrapolated rows exist", len(ext) > 0)
check("all extrapolated after last snapshot", (ext['Date'] > LAST).all() if len(ext) else True)

print("\
[D] FLAGGED vs TRUE SNAPSHOT SIZE (prediction: shortfall <= 5, recent only)")
snap_dates = np.sort(mem['snap_date'].unique())
rows = []
for sd in snap_dates:
    sd = pd.Timestamp(sd)
    true_n = mem[mem['snap_date'] == sd]['isin'].nunique()
    flagged_n = flag[(flag['effective_snap_date'] == sd) & (flag['in_nifty500_flag'] == 1)]['ticker'].nunique()
    rows.append(dict(snap_date=sd.date(), true=true_n, flagged=flagged_n, shortfall=true_n-flagged_n))
cmp_df = pd.DataFrame(rows)
max_short = cmp_df['shortfall'].max()
early = cmp_df[(cmp_df['shortfall'] > 2) & (cmp_df['snap_date'] < pd.Timestamp('2025-01-01').date())]
check("max shortfall <= 5", max_short <= 5, f"max={max_short}")
check("no large early shortfall (broken ISIN join)", len(early) == 0)

print("\
[E] REAL INDEX CHANGES")
for tkr, note in [('HDFC.NS','delisted 2023-07-12'), ('RANBAXY.NS','semi-annual grid, ~2015-04')]:
    g = flag[flag['ticker'] == tkr].sort_values('Date')
    if len(g) and (g['in_nifty500_flag'] == 1).any():
        print(f"    {tkr:<12} ({note}): last flag==1 = {g[g['in_nifty500_flag']==1]['Date'].max().date()}")
h = flag[(flag['ticker']=='HDFC.NS') & (flag['in_nifty500_flag']==1)]
if len(h): check("HDFC last flag==1 on/before 2023-07-12", h['Date'].max() <= pd.Timestamp('2023-07-12'))

print("\
[F] TRANSITIONS CLUSTER ON SNAPSHOT DATES (type-safe)")
snap_ts = sorted(pd.Timestamp(d) for d in mem['snap_date'].unique())
tdays = pd.DatetimeIndex(sorted(pd.Timestamp(d) for d in flag['Date'].unique()))
valid_trans = set()
for sd in snap_ts:
    pos = tdays.searchsorted(sd, side='left')
    if pos < len(tdays): valid_trans.add(tdays[pos])
f2 = flag.sort_values(['ticker','Date']).copy()
f2['prev'] = f2.groupby('ticker')['in_nifty500_flag'].shift(1)
f2['prev_snap'] = f2.groupby('ticker')['effective_snap_date'].shift(1)
trans = f2[f2['prev'].notna() & f2['in_nifty500_flag'].notna() & (f2['prev'] != f2['in_nifty500_flag'])]
tdates = set(pd.Timestamp(d) for d in trans['Date'].unique())
off = tdates - valid_trans
# off-snapshot transitions are benign IF the effective snapshot changed across a price gap
off_benign = True
for od in off:
    hits = trans[trans['Date'] == od]
    for r in hits.itertuples():
        if pd.Timestamp(r.prev_snap) == pd.Timestamp(r.effective_snap_date):
            off_benign = False
check("transitions on snapshot days, or benign gap-artifacts", off_benign,
      f"{len(off)} off-snapshot date(s), all benign" if off else "0 off-snapshot dates")
print(f"    distinct transition dates: {len(tdates)} (<= 33 expected)")

print("\
[G] 2018 CARRY-FORWARD PROVENANCE")
cf = flag[flag['snapshot_source'].astype(str).str.contains('carried_forward', na=False)]
check("carry-forward rows tagged", len(cf) > 0, f"{len(cf):,} rows")

print("\
" + "=" * 78)
print("STAGE 4 DIAGNOSTIC: " + ("FAIL — " + ", ".join(FAILURES) if FAILURES else "PASS"))
print("=" * 78)

STAGE 4 DIAGNOSTIC
[A] INTEGRITY
  [PASS] row count matches price panel  — 3,332,879 vs 3,332,879
  [PASS] ticker count == 1,069
  [PASS] no duplicate (Date,ticker)
[B] TRI-STATE BOUNDARY (NaN strictly before 2010-06-30)
  [PASS] NaN flags exist
  [PASS] all NaN strictly before first snapshot
  [PASS] no NaN on/after first snapshot
[C] EXTRAPOLATION BOUNDARY (tagged strictly after 2026-06-01)
  [PASS] extrapolated rows exist
  [PASS] all extrapolated after last snapshot
[D] FLAGGED vs TRUE SNAPSHOT SIZE (prediction: shortfall <= 5, recent only)
  [PASS] max shortfall <= 5  — max=4
  [PASS] no large early shortfall (broken ISIN join)
[E] REAL INDEX CHANGES
    HDFC.NS      (delisted 2023-07-12): last flag==1 = 2023-07-12
    RANBAXY.NS   (semi-annual grid, ~2015-04): last flag==1 = 2014-12-30
  [PASS] HDFC last flag==1 on/before 2023-07-12
[F] TRANSITIONS CLUSTER ON SNAPSHOT DATES (type-safe)
  [PASS] transitions on snapshot days, or benign gap-artifacts  — 2 off-snapshot date(s), all b

## Stage 3B — Remediate the raw price panel  (build v3)

Two remediation passes, producing **v3** from v1.

**Part 1 — per-ticker defect masking** (unchanged from the v2 build). Masks phantom rows, split splices, and bad prints. *Only OHLC is set to NaN; Volume and the row are retained.*

**Part 2 — market-closed date removal** (new). yfinance fabricates a full row for every ticker on NSE holidays. Part 1 correctly masks each one but leaves the row in place — and a masked row means *"price unknown"*, whereas a holiday means *"no market existed"*. Carrying the second as the first kills **two** returns per holiday (the holiday itself, and the next real session, which has no prior close) for **every ticker simultaneously**.

Detection is structural, not a hardcoded date list:

```
market_closed(date) := masked_fraction >= 0.95  AND  zero rows with Volume > 0
```

Both conditions are required. The first alone cannot distinguish a closed market from a **vendor failure** — a live trading day where yfinance simply failed to deliver. `2025-03-18` is the counterexample: 98% masked, but the 19 survivors show real trading (ITC 13.9M shares), and NSE's circular confirms the market was open. It is **retained and tagged** via the new `date_status` column, not deleted.

A guard asserts the rule reproduces the six investigated closures and the one vendor failure before any row is deleted.

In [5]:
# =============================================================================
# STAGE 3B — REMEDIATION (build v3 from v1)   [requires Cell A]
#
# Idempotent: refuses to overwrite a v3 built today. To re-read an existing v3,
# skip this cell and run the diagnostic.
#
# THIS CELL DOES TWO THINGS:
#
#   PART 1 (unchanged from the v2 build): mask three per-ticker defect classes
#     -- phantom rows, split splices, bad prints. OHLC -> NaN, Volume and the
#     row RETAINED. This is fact #14 remediation and it is correct.
#
#   PART 2 (NEW): delete rows on MARKET-CLOSED DATES.
#
# WHY PART 2 EXISTS
# -----------------
# Part 1 detects phantoms PER TICKER: "this stock did not trade today." It never
# asks whether EVERY ticker is phantom on the same date. When 931 of 931 rows on
# a date are phantom, that is not 931 independent no-trade events -- it is a
# CLOSED EXCHANGE, and the row should not exist at all.
#
# yfinance fabricates a full row for NSE holidays (prior close copied into OHLC,
# Volume 0). Part 1 correctly masked each one, but left the rows in place. A
# masked row means "price unknown"; a holiday means "no market existed". Carrying
# the second as the first breaks every rolling window that spans it:
#   - the holiday row has no return (no close), AND
#   - the NEXT real trading day has no return either (no prior close).
# One fabricated holiday kills TWO returns, for EVERY ticker simultaneously.
#
# Measured impact before this fix: 2025 momentum-eligible universe collapsed
# from ~480 names to 126, and 2026 to 22, because every stock's 232-day window
# spanned a fabricated holiday.
#
# DETECTION IS STRUCTURAL, NOT A HARDCODED DATE LIST
# --------------------------------------------------
#   market_closed(date) := (masked_fraction >= 0.95) AND (zero rows with Volume > 0)
#
# BOTH conditions are required. The first alone CANNOT distinguish a closed
# market from a VENDOR FAILURE -- a live trading day where yfinance simply failed
# to deliver data. 2025-03-18 is the counterexample that proves it: 98% of rows
# were phantom-masked, but the 19 survivors show real trading (ITC 13.9M shares,
# SpiceJet 13.3M). NSE's published circular confirms the market was OPEN that day.
# Deleting it would destroy a real trading day.
#
# So: closed dates are DELETED; vendor-failure dates are RETAINED and TAGGED via
# the new `date_status` column, so downstream coverage filters can exclude them
# from a denominator without pretending the day did not happen.
#
# The four 2026 dates detected here are independently confirmed against NSE's
# published 2026 trading-holiday circular (Maharashtra municipal elections,
# Maharashtra Day, Bakri Eid, Moharram).
# =============================================================================
import numpy as np

V1 = resolve('price_panel_v1')
try:
    OUT_V3 = out_path('price_panel_v3')
    DO_BUILD = True
except FileExistsError as e:
    print(f"v3 already exists today — skipping build. ({e})")
    print("Run the diagnostic cell; it reads the newest v3 automatically.")
    DO_BUILD = False

if DO_BUILD:
    CHANGE_LOG   = BASE / 'validation' / f'remediation_change_log_{BUILD_DATE}.csv'
    CLOSED_LOG   = BASE / 'validation' / f'market_closed_dates_removed_{BUILD_DATE}.csv'
    print("=" * 78); print("STAGE 3B — REMEDIATION (build v3)"); print("=" * 78)

    px = pd.read_parquet(V1); px['Date'] = pd.to_datetime(px['Date'])
    px = px.sort_values(['ticker','Date']).reset_index(drop=True)
    V1_ROWS = len(px)
    assert V1_ROWS == 3_332_879 and px['ticker'].nunique() == 1069, "v1 shape wrong. STOP."

    # -------------------------------------------------------------------------
    # PART 1 — per-ticker defect masking (identical to the v2 build)
    # -------------------------------------------------------------------------
    print("\
--- PART 1: per-ticker defect masking ---")

    # Mechanism 1: phantom rows
    px['_phantom'] = ((px['Open']==px['High']) & (px['High']==px['Low']) &
                      (px['Low']==px['Close']) & px['Open'].notna() & (px['Volume']==0))
    # Mechanism 2: split splices (8 named)
    SPLICE = [('GREENPLY.NS','2016-01-06'),('MVL.NS','2010-11-16'),('TTML.NS','2013-08-07'),
              ('IGARASHI.NS','2018-09-27'),('KTKBANK.NS','2020-03-17'),('KARURVYSYA.NS','2018-08-14'),
              ('JISLJALEQS.NS','2011-11-08'),('AJRINFRA.NS','2013-01-23')]
    splice_keys = {(t, pd.Timestamp(d)) for t,d in SPLICE}
    px['_splice'] = [(t,d) in splice_keys for t,d in zip(px['ticker'], px['Date'])]
    # Mechanism 3: bad prints (6 named)
    PRINTS = [('THYROCARE.NS','2019-10-27'),('THYROCARE.NS','2020-11-14'),('INDIAMART.NS','2019-10-27'),
              ('INDIAMART.NS','2020-11-14'),('MVL.NS','2019-10-10'),('MVL.NS','2019-12-20')]
    print_keys = {(t, pd.Timestamp(d)) for t,d in PRINTS}
    px['_badprint'] = [(t,d) in print_keys for t,d in zip(px['ticker'], px['Date'])]

    n_ph, n_sp, n_bp = int(px['_phantom'].sum()), int(px['_splice'].sum()), int(px['_badprint'].sum())
    print(f"  phantom: {n_ph:,} (exp 57,876)  splice: {n_sp} (exp 8)  badprint: {n_bp} (exp 6)")
    assert n_ph == 57876, f"phantom count {n_ph} != 57,876 — panel differs from diagnostic. STOP."
    assert n_sp == 8 and n_bp == 6, "splice/badprint count wrong. STOP."
    bbg_hit = px[(px['_phantom']|px['_splice']|px['_badprint']) & (px['source']=='bloomberg')]
    assert len(bbg_hit) == 0, "Bloomberg rows flagged — unexpected. STOP."

    px['_to_mask'] = px['_phantom'] | px['_splice'] | px['_badprint']
    px['price_status'] = 'observed'
    px.loc[px['_phantom'], 'price_status'] = 'phantom_masked'
    px.loc[px['_splice'], 'price_status'] = 'split_splice_masked'
    px.loc[px['_badprint'], 'price_status'] = 'bad_print_masked'

    log = px.loc[px['_to_mask'], ['Date','ticker','Open','High','Low','Close','Volume','source','price_status']].copy()
    log = log.rename(columns={'Open':'orig_Open','High':'orig_High','Low':'orig_Low','Close':'orig_Close'})
    log['action'] = 'OHLC_set_to_NaN'; log['volume_retained'] = True

    for col in ['Open','High','Low','Close']:
        px.loc[px['_to_mask'], col] = np.nan

    resid = ((px['Open']==px['High']) & (px['High']==px['Low']) & (px['Low']==px['Close']) &
             px['Open'].notna() & (px['Volume']==0))
    assert resid.sum() == 0, "Phantom rows survived. STOP."
    assert len(px[px['_to_mask'] & px[['Open','High','Low','Close']].notna().any(axis=1)]) == 0, "Masked row kept a price. STOP."
    assert len(px) == V1_ROWS, "Part 1 changed the row count — it must only mask. STOP."
    print(f"  masked {int(px['_to_mask'].sum()):,} rows (OHLC -> NaN, Volume + row retained)")

    # -------------------------------------------------------------------------
    # PART 2 — market-closed date detection and removal
    # -------------------------------------------------------------------------
    print("\
--- PART 2: market-closed date detection ---")

    MASKED_FRAC_MIN = 0.95   # a closed date is ~entirely fabricated
    REAL_TRADE_MAX  = 0      # and has ZERO rows showing real volume

    per_date = px.groupby('Date').agg(
        n_rows        = ('ticker', 'size'),
        n_masked      = ('price_status', lambda s: int((s != 'observed').sum())),
        n_real_volume = ('Volume', lambda s: int((s.fillna(0) > 0).sum())),
    )
    per_date['masked_frac'] = per_date['n_masked'] / per_date['n_rows']

    is_closed = ((per_date['masked_frac'] >= MASKED_FRAC_MIN) &
                 (per_date['n_real_volume'] <= REAL_TRADE_MAX))
    closed_dates = per_date.index[is_closed]

    # High-masked dates that FAIL the volume test = vendor failures, NOT closures.
    is_vendor_fail = ((per_date['masked_frac'] >= MASKED_FRAC_MIN) &
                      (per_date['n_real_volume'] > REAL_TRADE_MAX))
    vendor_fail_dates = per_date.index[is_vendor_fail]

    print(f"  Dates with masked_frac >= {MASKED_FRAC_MIN:.0%}: {int((per_date['masked_frac'] >= MASKED_FRAC_MIN).sum())}")
    print(f"    of which MARKET CLOSED (zero real volume) : {len(closed_dates)}")
    print(f"    of which VENDOR FAILURE (real trades seen) : {len(vendor_fail_dates)}")

    # -- Only whole-universe closures are removed. --------------------------------
    # 139 of the closed dates carry a HANDFUL of rows (n=1..18) -- real NSE
    # holidays where yfinance fabricated a row for only a few tickers. They are
    # detected correctly but cause no measurable coverage damage, and removing
    # them would touch far more of the panel for no benefit. DELIBERATE DECISION:
    # they are RETAINED and recorded as a documented limitation. Only dates
    # affecting a large share of the universe are removed.
    UNIVERSE_SHARE_MIN = 0.50
    active_per_date = per_date['n_rows']
    median_rows = float(active_per_date.median())
    big_closed = [d for d in closed_dates
                  if per_date.loc[d, 'n_rows'] >= UNIVERSE_SHARE_MIN * median_rows]
    small_closed = [d for d in closed_dates if d not in set(big_closed)]

    print(f"\
  Median rows per date: {median_rows:,.0f}")
    print(f"  Closed dates affecting >= {UNIVERSE_SHARE_MIN:.0%} of universe -> REMOVE: {len(big_closed)}")
    print(f"  Closed dates affecting a handful of tickers -> RETAIN (documented): {len(small_closed)}")

    print("\
  MARKET-CLOSED DATES TO BE REMOVED:")
    print(f"    {'date':<14} {'rows':>7} {'masked':>8} {'masked%':>9} {'real_vol_rows':>15}")
    for d in sorted(big_closed):
        row = per_date.loc[d]
        print(f"    {str(d.date()):<14} {int(row['n_rows']):>7,} {int(row['n_masked']):>8,} "
              f"{100*row['masked_frac']:>8.1f}% {int(row['n_real_volume']):>15}")

    if len(vendor_fail_dates):
        print("\
  VENDOR-FAILURE DATES (RETAINED — market was OPEN, data missing):")
        print(f"    {'date':<14} {'rows':>7} {'masked%':>9} {'real_vol_rows':>15}")
        for d in sorted(vendor_fail_dates):
            row = per_date.loc[d]
            print(f"    {str(d.date()):<14} {int(row['n_rows']):>7,} "
                  f"{100*row['masked_frac']:>8.1f}% {int(row['n_real_volume']):>15}")

    # -- GUARD: detection must reproduce the known answer. ------------------------
    # This notebook was built after a manual investigation that identified six
    # whole-universe closures and one vendor failure. If the structural rule does
    # not reproduce that, the rule is wrong and we must NOT delete anything.
    EXPECTED_CLOSED = {pd.Timestamp(d) for d in
                       ['2014-04-24','2014-10-15','2026-01-15','2026-05-01',
                        '2026-05-28','2026-06-26']}
    EXPECTED_VENDOR_FAIL = {pd.Timestamp('2025-03-18')}

    got_closed = set(big_closed)
    got_vendor = set(vendor_fail_dates)

    if got_closed != EXPECTED_CLOSED:
        print("\
  !! DETECTION DISAGREES WITH THE INVESTIGATED ANSWER !!")
        print(f"     unexpected extra : {sorted(str(d.date()) for d in got_closed - EXPECTED_CLOSED)}")
        print(f"     expected missing : {sorted(str(d.date()) for d in EXPECTED_CLOSED - got_closed)}")
        raise AssertionError(
            "Market-closed detection did not reproduce the six investigated dates. "
            "Do NOT delete rows on an unvalidated rule. Investigate the difference "
            "first -- an extra date may be a genuine new finding, a missing one "
            "means the rule is broken.")
    assert got_vendor == EXPECTED_VENDOR_FAIL, (
        f"Vendor-failure detection returned {sorted(str(d.date()) for d in got_vendor)}, "
        f"expected 2025-03-18. STOP.")
    print("\
  [PASS] detection reproduces the six investigated closures + one vendor failure")

    # -- Log before deleting. -----------------------------------------------------
    closed_log = per_date.loc[sorted(big_closed)].reset_index()
    closed_log['classification']   = 'market_closed'
    closed_log['action']           = 'rows_deleted'
    closed_log['detection_rule']   = f'masked_frac>={MASKED_FRAC_MIN} AND n_real_volume<={REAL_TRADE_MAX}'
    if len(vendor_fail_dates):
        vf = per_date.loc[sorted(vendor_fail_dates)].reset_index()
        vf['classification'] = 'vendor_failure'
        vf['action']         = 'rows_retained_tagged'
        vf['detection_rule'] = f'masked_frac>={MASKED_FRAC_MIN} AND n_real_volume>{REAL_TRADE_MAX}'
        closed_log = pd.concat([closed_log, vf], ignore_index=True)
    closed_log.to_csv(CLOSED_LOG, index=False)

    # -- date_status column, set BEFORE deletion so the tag survives. -------------
    px['date_status'] = 'normal'
    px.loc[px['Date'].isin(vendor_fail_dates), 'date_status'] = 'vendor_failure'

    n_before = len(px)
    px = px[~px['Date'].isin(big_closed)].reset_index(drop=True)
    n_removed = n_before - len(px)
    print(f"\
  Removed {n_removed:,} rows across {len(big_closed)} market-closed dates")
    print(f"  Rows: {n_before:,} -> {len(px):,}")
    print(f"  Distinct dates: {len(per_date):,} -> {px['Date'].nunique():,}")
    print(f"  date_status: {dict(px['date_status'].value_counts())}")

    # -- Post-deletion invariants. ------------------------------------------------
    assert not px['Date'].isin(big_closed).any(), "A closed date survived deletion. STOP."
    assert px['Date'].isin(vendor_fail_dates).any(), "Vendor-failure date was wrongly deleted. STOP."
    assert px['ticker'].nunique() == 1069, "Ticker count changed — deletion removed a whole ticker. STOP."
    assert px.duplicated(['ticker','Date']).sum() == 0, "Duplicate keys after deletion. STOP."

    out_cols = ['Date','ticker','Open','High','Low','Close','Volume',
                'adj_close_dividend_yfinance_unused','source','price_status','date_status']
    out_cols = [c for c in out_cols if c in px.columns]
    px[out_cols].to_parquet(OUT_V3, index=False)
    log.sort_values(['ticker','Date']).to_csv(CHANGE_LOG, index=False)
    print(f"\
  -> written v3: {OUT_V3.name}")
    print(f"  -> mask change log : {CHANGE_LOG.name} ({len(log):,} rows)")
    print(f"  -> closed-date log : {CLOSED_LOG.name} ({len(closed_log)} dates)")
    print(f"  price_status: {dict(px['price_status'].value_counts())}")
    print("\
  !! RE-RUN CELL A NOW so EXPECTED_ROWS picks up the new v3 row count !!")
    print("=" * 78); print("STAGE 3B BUILD COMPLETE — re-run Cell A, then the diagnostic."); print("=" * 78)


STAGE 3B — REMEDIATION (build v3)
--- PART 1: per-ticker defect masking ---
  phantom: 57,876 (exp 57,876)  splice: 8 (exp 8)  badprint: 6 (exp 6)
  masked 57,890 rows (OHLC -> NaN, Volume + row retained)
--- PART 2: market-closed date detection ---
  Dates with masked_frac >= 95%: 146
    of which MARKET CLOSED (zero real volume) : 145
    of which VENDOR FAILURE (real trades seen) : 1
  Median rows per date: 817
  Closed dates affecting >= 50% of universe -> REMOVE: 6
  Closed dates affecting a handful of tickers -> RETAIN (documented): 139
  MARKET-CLOSED DATES TO BE REMOVED:
    date              rows   masked   masked%   real_vol_rows
    2014-04-24         668      668    100.0%               0
    2014-10-15         671      671    100.0%               0
    2026-01-15         769      769    100.0%               0
    2026-05-01         777      777    100.0%               0
    2026-05-28         931      931    100.0%               0
    2026-06-26         931      931    100

### Stage 3B — diagnostic *(gate)*

Confirms v2 is structurally identical to v1 except where intended: zero phantoms survive, Bloomberg rows bit-identical, every named splice/bad-print masked, untouched rows bit-identical, and the usable universe (≥120 valid Close) unchanged.

In [6]:
# =============================================================================
# STAGE 3B — DIAGNOSTIC (gate)   READ-ONLY
#
# Compares v1 against v3. Unlike the old v1-vs-v2 diagnostic, the row counts
# now DIFFER BY DESIGN: v3 deletes market-closed dates. So instead of asserting
# "row count identical", this asserts the exact arithmetic --
#   len(v3) == len(v1) - (rows on the removed dates)
# -- and verifies that v1 and v3 are otherwise key-identical outside those dates.
# =============================================================================
import numpy as np

v1 = pd.read_parquet(resolve('price_panel_v1')); v1['Date'] = pd.to_datetime(v1['Date'])
v1 = v1.sort_values(['ticker','Date']).reset_index(drop=True)
v3 = pd.read_parquet(resolve(RAW_PANEL_KEY)); v3['Date'] = pd.to_datetime(v3['Date'])
v3 = v3.sort_values(['ticker','Date']).reset_index(drop=True)

print("=" * 78); print(f"STAGE 3B DIAGNOSTIC — {resolve(RAW_PANEL_KEY).name}"); print("=" * 78)
FAILURES = []
def check(name, cond, detail=""):
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}" + (f"  — {detail}" if detail else ""))
    if not cond: FAILURES.append(name)

REMOVED_DATES = sorted(set(v1['Date'].unique()) - set(v3['Date'].unique()))
REMOVED_DATES = [pd.Timestamp(d) for d in REMOVED_DATES]
v1_on_removed = v1[v1['Date'].isin(REMOVED_DATES)]

print("\
[A] DELETION ARITHMETIC")
print(f"    dates removed: {len(REMOVED_DATES)} -> {[str(d.date()) for d in REMOVED_DATES]}")
check("exactly 6 dates removed", len(REMOVED_DATES) == 6, f"{len(REMOVED_DATES)}")
check("row arithmetic exact", len(v3) == len(v1) - len(v1_on_removed),
      f"{len(v1):,} - {len(v1_on_removed):,} = {len(v1) - len(v1_on_removed):,} vs actual {len(v3):,}")
check("ticker count == 1,069", v3['ticker'].nunique() == 1069)
check("price_status present", 'price_status' in v3.columns)
check("date_status present", 'date_status' in v3.columns)
check("no duplicate (Date,ticker)", not v3.duplicated(['Date','ticker']).any())

print("\
[B] REMOVED DATES WERE GENUINELY CLOSED")
# Every removed date must have had ZERO real-volume rows in v1. If any removed
# date shows real trading, we deleted a live session -- the worst possible error.
for d in REMOVED_DATES:
    sub = v1[v1['Date'] == d]
    n_realvol = int((sub['Volume'].fillna(0) > 0).sum())
    check(f"{d.date()} had zero real-volume rows in v1", n_realvol == 0, f"{n_realvol} rows traded")

print("\
[C] VENDOR-FAILURE DATE RETAINED")
vf = v3[v3['date_status'] == 'vendor_failure']
check("vendor_failure rows present in v3", len(vf) > 0, f"{len(vf):,} rows")
if len(vf):
    vf_dates = sorted(pd.to_datetime(vf['Date'].unique()))
    print(f"    dates: {[str(pd.Timestamp(d).date()) for d in vf_dates]}")
    check("2025-03-18 retained", pd.Timestamp('2025-03-18') in set(vf_dates))
    real = v3[(v3['Date'] == pd.Timestamp('2025-03-18')) & (v3['Volume'].fillna(0) > 0)]
    check("2025-03-18 still shows real trading (proves it was open)", len(real) > 0, f"{len(real)} rows")

print("\
[D] EVERYTHING OUTSIDE REMOVED DATES IS UNCHANGED")
v1_keep = v1[~v1['Date'].isin(REMOVED_DATES)].sort_values(['ticker','Date']).reset_index(drop=True)
check("(Date,ticker) keys identical outside removed dates",
      v1_keep[['Date','ticker']].equals(v3[['Date','ticker']]))
check("Volume identical outside removed dates", v1_keep['Volume'].equals(v3['Volume']))

print("\
[E] PHANTOM ERADICATION")
resid = ((v3['Open']==v3['High']) & (v3['High']==v3['Low']) & (v3['Low']==v3['Close']) &
         v3['Open'].notna() & (v3['Volume']==0))
check("zero phantom rows remain", int(resid.sum()) == 0, f"{int(resid.sum())}")
untouched = v3['price_status'] == 'observed'
for col in ['Open','High','Low','Close']:
    check(f"{col} bit-identical on untouched rows",
          v1_keep.loc[untouched.values,col].equals(v3.loc[untouched.values,col]))

print("\
[F] BLOOMBERG UNTOUCHED")
b1 = v1_keep[v1_keep['source']=='bloomberg'].reset_index(drop=True)
b3 = v3[v3['source']=='bloomberg'].reset_index(drop=True)
check("Bloomberg row count unchanged outside removed dates", len(b1) == len(b3), f"{len(b3):,}")
for col in ['Open','High','Low','Close','Volume']:
    check(f"Bloomberg {col} bit-identical", b1[col].equals(b3[col]))

print("\
[G] NAMED EVENTS MASKED")
SPLICE = [('GREENPLY.NS','2016-01-06'),('MVL.NS','2010-11-16'),('TTML.NS','2013-08-07'),
          ('IGARASHI.NS','2018-09-27'),('KTKBANK.NS','2020-03-17'),('KARURVYSYA.NS','2018-08-14'),
          ('JISLJALEQS.NS','2011-11-08'),('AJRINFRA.NS','2013-01-23')]
PRINTS = [('THYROCARE.NS','2019-10-27'),('THYROCARE.NS','2020-11-14'),('INDIAMART.NS','2019-10-27'),
          ('INDIAMART.NS','2020-11-14'),('MVL.NS','2019-10-10'),('MVL.NS','2019-12-20')]
for label, events in [("splice", SPLICE), ("bad print", PRINTS)]:
    ok = sum(1 for t,d in events if len(v3[(v3['ticker']==t) & (v3['Date']==pd.Timestamp(d))]) == 1
             and v3[(v3['ticker']==t) & (v3['Date']==pd.Timestamp(d))].iloc[0][['Open','High','Low','Close']].isna().all())
    check(f"all {len(events)} {label} rows masked", ok == len(events), f"{ok}/{len(events)}")
print(f"    price_status: {dict(v3['price_status'].value_counts())}")

print("\
[H] USABLE UNIVERSE UNCHANGED (>=120 valid Close)")
u1 = v1.groupby('ticker')['Close'].apply(lambda s: s.notna().sum())
u3 = v3.groupby('ticker')['Close'].apply(lambda s: s.notna().sum())
check("usable-ticker count unchanged", (u1>=120).sum() == (u3>=120).sum(),
      f"{(u1>=120).sum()} -> {(u3>=120).sum()}")

print("\
[I] NO CALENDAR GAP INTRODUCED")
# Deleting a date adjacent to a weekend could create a >4-day hole. The only
# legitimate long gap in this panel is 2022-04-18 (Good Friday, Thu->Mon).
dts = pd.Series(sorted(pd.to_datetime(v3['Date'].unique())))
gaps = dts.diff().dt.days
big = pd.DataFrame({'date': dts[1:], 'gap': gaps[1:]}).query('gap > 4')
print(f"    gaps > 4 calendar days: {len(big)}")
for _, g in big.iterrows():
    print(f"      {g['date'].date()}  ({int(g['gap'])} days)")
check("no more than 2 long gaps (2022-04-18 Good Friday is expected)", len(big) <= 2, f"{len(big)}")

print("\
" + "=" * 78)
print("STAGE 3B DIAGNOSTIC: " + ("FAIL — " + ", ".join(FAILURES) if FAILURES else "PASS"))
if not FAILURES:
    print(f"  v3 = {len(v3):,} rows, {v3['Date'].nunique():,} dates")
    print("  If you have not re-run Cell A since the build, DO IT NOW.")
print("=" * 78)


STAGE 3B DIAGNOSTIC — raw_price_panel_consolidated_v3_2026-07-24.parquet
[A] DELETION ARITHMETIC
    dates removed: 6 -> ['2014-04-24', '2014-10-15', '2026-01-15', '2026-05-01', '2026-05-28', '2026-06-26']
  [PASS] exactly 6 dates removed  — 6
  [PASS] row arithmetic exact  — 3,332,879 - 4,747 = 3,328,132 vs actual 3,328,132
  [PASS] ticker count == 1,069
  [PASS] price_status present
  [PASS] date_status present
  [PASS] no duplicate (Date,ticker)
[B] REMOVED DATES WERE GENUINELY CLOSED
  [PASS] 2014-04-24 had zero real-volume rows in v1  — 0 rows traded
  [PASS] 2014-10-15 had zero real-volume rows in v1  — 0 rows traded
  [PASS] 2026-01-15 had zero real-volume rows in v1  — 0 rows traded
  [PASS] 2026-05-01 had zero real-volume rows in v1  — 0 rows traded
  [PASS] 2026-05-28 had zero real-volume rows in v1  — 0 rows traded
  [PASS] 2026-06-26 had zero real-volume rows in v1  — 0 rows traded
[C] VENDOR-FAILURE DATE RETAINED
  [PASS] vendor_failure rows present in v3  — 915 rows
    d

## Stage 5a — Returns  (build from v3)

Three returns on the remediated v3 panel: `overnight = open[t]/close[t-1]-1`, `intraday = close[t]/open[t]-1`, `daily = close[t]/close[t-1]-1`.

`prev_close` is `groupby('ticker')['_close'].shift(1)` — a shift by **row**, not by date. This is why the market-closed-date deletion needs no logic change here: once the fabricated holiday rows are gone, `shift(1)` on the next session automatically lands on the previous *real* trading day, and the returns that were NaN in v2 compute themselves.

In [7]:
# =============================================================================
# STAGE 5a — RETURNS (build from v3)   [requires Cell A]
# =============================================================================
import numpy as np

try:
    OUT = out_path('returns_panel_v2'); DO_BUILD = True
except FileExistsError as e:
    print(f"returns_panel_v2 exists today — skipping build. ({e})"); DO_BUILD = False

if DO_BUILD:
    print("=" * 78); print("STAGE 5a — RETURNS (from v3)"); print("=" * 78)
    px = pd.read_parquet(resolve(RAW_PANEL_KEY)); px['Date'] = pd.to_datetime(px['Date'])
    px = px.sort_values(['ticker','Date']).reset_index(drop=True)
    assert px['ticker'].nunique() == 1069 and 'price_status' in px.columns, "v3 wrong. STOP."
    assert px.duplicated(['ticker','Date']).sum() == 0, "Duplicate keys. STOP."

    px['_open']  = px['Open'].where(px['Open'] > 0)
    px['_close'] = px['Close'].where(px['Close'] > 0)
    px['prev_close'] = px.groupby('ticker')['_close'].shift(1)
    px['prev_date']  = px.groupby('ticker')['Date'].shift(1)
    px['days_since_prev_trade'] = (px['Date'] - px['prev_date']).dt.days

    px['overnight_return'] = px['_open']  / px['prev_close'] - 1
    px['intraday_return']  = px['_close'] / px['_open']      - 1
    px['daily_return']     = px['_close'] / px['prev_close'] - 1
    for c in ['overnight_return','intraday_return','daily_return']:
        assert not np.isinf(px[c]).any(), f"{c} has inf. STOP."
        print(f"  {c:<18}: {px[c].notna().sum():>9,} non-null")

    cols = ['Date','ticker','Open','High','Low','Close','Volume','source','price_status',
            'date_status',
            'prev_close','prev_date','days_since_prev_trade',
            'overnight_return','intraday_return','daily_return']
    cols = [c for c in cols if c in px.columns]
    out = px[cols].copy()
    assert_rows(len(out), "Stage 5a returns_panel")
    out.to_parquet(OUT, index=False)
    print(f"\
  -> written: {OUT.name}")
    print("=" * 78); print("STAGE 5a BUILD COMPLETE — run the diagnostic."); print("=" * 78)

  NOTE: 4 older-dated 'returns_panel_v2' file(s) exist; reads use the newest, which will be returns_panel_v2_2026-07-24.parquet after this build.
STAGE 5a — RETURNS (from v3)
  overnight_return  : 3,252,207 non-null
  intraday_return   : 3,259,329 non-null
  daily_return      : 3,251,675 non-null
  [SKIP] Stage 5a returns_panel row-count assert -- EXPECTED_ROWS unset (raw panel not built; re-run Cell A after Stage 3B)
  -> written: returns_panel_v2_2026-07-24.parquet
STAGE 5a BUILD COMPLETE — run the diagnostic.


### Stage 5a — diagnostic *(gate)*

The identity `(1+overnight)(1+intraday)-1 == daily` must hold to ~1e-15 on every row (catches any misalignment). Plus mask propagation (masked rows and successors are NaN), coverage by year, and the ±30% review list (flagged, never dropped).

In [11]:
# =============================================================================
# STAGE 5a — DIAGNOSTIC (gate)   Writes the ±30% review CSV only.
# =============================================================================
import numpy as np

r = pd.read_parquet(resolve('returns_panel_v2'))
r['Date'] = pd.to_datetime(r['Date']); r['prev_date'] = pd.to_datetime(r['prev_date'])
OUT_EXT = BASE / 'validation' / f'daily_return_beyond_30pct_v2_{BUILD_DATE}.csv'

print("=" * 78); print("STAGE 5a DIAGNOSTIC"); print("=" * 78)
FAILURES = []
def check(name, cond, detail=""):
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}" + (f"  — {detail}" if detail else ""))
    if not cond: FAILURES.append(name)

print("\
[A] INTEGRITY")
check(f"row count == raw panel ({EXPECTED_ROWS:,})", len(r) == EXPECTED_ROWS)
check("ticker count == 1,069", r['ticker'].nunique() == 1069)
for c in ['overnight_return','intraday_return','daily_return']:
    check(f"{c} no inf", not np.isinf(r[c]).any())

print("\
[B] IDENTITY (1+on)(1+id)-1 == daily")
m = r[['overnight_return','intraday_return','daily_return']].notna().all(axis=1)
sub = r[m]
resid = ((1+sub['overnight_return'])*(1+sub['intraday_return'])-1 - sub['daily_return']).abs()
print(f"  rows all three non-null: {len(sub):,}   max|residual|: {resid.max():.2e}")
check("identity holds to 1e-9", (resid > 1e-9).sum() == 0, f"{(resid>1e-9).sum()} violations")

print("\
[C] MASK PROPAGATION")
masked = r['price_status'] != 'observed'
own = r[masked][['overnight_return','intraday_return','daily_return']].notna().any(axis=1)
check("no masked row carries a return", int(own.sum()) == 0)
r['_prev_status'] = r.groupby('ticker')['price_status'].shift(1)
succ = r[(r['_prev_status'].notna()) & (r['_prev_status'] != 'observed')]
check("no successor of a masked row carries daily_return", succ['daily_return'].notna().sum() == 0)

print("\
[D] COVERAGE BY YEAR")
r['_yr'] = r['Date'].dt.year
cov = r.groupby('_yr').agg(rows=('ticker','size'),
    overnight=('overnight_return', lambda s: 100*s.notna().mean()),
    daily=('daily_return', lambda s: 100*s.notna().mean()))
print(cov.to_string(float_format=lambda x: f"{x:,.2f}"))

print("\
[E] ±30% REVIEW LIST (flagged, not dropped)")
ext = r[r['daily_return'].abs() > 0.30].copy()
print(f"  rows beyond ±30%: {len(ext):,}   distinct tickers: {ext['ticker'].nunique()}")
# NOTE: this band was calibrated on v2. v3 RECOVERS returns that were
# previously NaN (the trading day after a deleted holiday now computes against
# the previous REAL session), so a modest increase here is EXPECTED, not a
# regression. Band widened; re-baseline once the v3 number is known.
check("list is non-trivial and bounded (real extremes remain)", 50 <= len(ext) <= 600, f"{len(ext)}")
eo = ext[['Date','ticker','prev_date','days_since_prev_trade','prev_close','Open','High','Low',
          'Close','Volume','overnight_return','intraday_return','daily_return','price_status','source']].copy()
eo['flag_reason'] = 'daily_return_magnitude_gt_30pct'; eo['review_status'] = 'pending_manual_review'
eo.sort_values('Date').to_csv(OUT_EXT, index=False)
print(f"  -> logged {OUT_EXT.name}")

print("\
" + "=" * 78)
print("STAGE 5a DIAGNOSTIC: " + ("FAIL — " + ", ".join(FAILURES) if FAILURES else "PASS"))
print("=" * 78)

STAGE 5a DIAGNOSTIC
[A] INTEGRITY
  [PASS] row count == raw panel (3,328,132)
  [PASS] ticker count == 1,069
  [PASS] overnight_return no inf
  [PASS] intraday_return no inf
  [PASS] daily_return no inf
[B] IDENTITY (1+on)(1+id)-1 == daily
  rows all three non-null: 3,251,675   max|residual|: 7.11e-15
  [PASS] identity holds to 1e-9  — 0 violations
[C] MASK PROPAGATION
  [PASS] no masked row carries a return
  [PASS] no successor of a masked row carries daily_return
[D] COVERAGE BY YEAR
        rows  overnight  daily
_yr                           
2010  178551      97.46  97.46
2011  182428      98.27  98.27
2012  186033      98.14  98.14
2013  189456      97.67  97.67
2014  185025      98.36  98.36
2015  189485      97.94  97.94
2016  193184      97.95  97.95
2017  198979      98.07  98.07
2018  202672      98.36  98.36
2019  200625      98.41  98.41
2020  208577      97.95  97.95
2021  210997      98.02  98.02
2022  217166      97.77  97.77
2023  216525      97.17  97.17
2024  221084

## Stage 5b — Turnover + realized volatility  (build)

`turnover = volume*close`; `realized_vol_60/120 = rolling std of daily_return (full window, ddof=1) * sqrt(252)`. Full window (min_periods=window) means vol first appears on a ticker's day 61/121 — never backfilled. A window containing any NaN daily_return yields NaN (strict), so masked gaps don't leak into vol.

In [12]:
# =============================================================================
# STAGE 5b — TURNOVER + REALIZED VOL (build)   [requires Cell A]
# =============================================================================
import numpy as np

try:
    OUT = out_path('features_daily_v2'); DO_BUILD = True
except FileExistsError as e:
    print(f"features_daily_v2 exists today — skipping build. ({e})"); DO_BUILD = False

if DO_BUILD:
    W_S, W_L, ANN, DDOF = 60, 120, np.sqrt(252), 1
    print("=" * 78); print("STAGE 5b — TURNOVER + REALIZED VOL"); print("=" * 78)
    r = pd.read_parquet(resolve('returns_panel_v2')); r['Date'] = pd.to_datetime(r['Date'])
    r = r.sort_values(['ticker','Date']).reset_index(drop=True)
    assert r['ticker'].nunique() == 1069, "wrong input. STOP."

    r['turnover'] = r['Volume'] * r['Close']
    assert (r['turnover'].dropna() >= 0).all(), "Negative turnover. STOP."
    print(f"  turnover non-null: {r['turnover'].notna().sum():,}")

    def rvol(s, w): return s.rolling(window=w, min_periods=w).std(ddof=DDOF)
    grp = r.groupby('ticker')['daily_return']
    r['realized_vol_60']  = grp.transform(lambda s: rvol(s, W_S)) * ANN
    r['realized_vol_120'] = grp.transform(lambda s: rvol(s, W_L)) * ANN
    for c in ['realized_vol_60','realized_vol_120']:
        assert not np.isinf(r[c]).any() and (r[c].dropna() >= 0).all(), f"{c} bad. STOP."
        print(f"  {c:<18}: {r[c].notna().sum():>9,} non-null")

    cols = ['Date','ticker','Open','High','Low','Close','Volume','source','price_status',
            'date_status',
            'prev_close','prev_date','days_since_prev_trade',
            'overnight_return','intraday_return','daily_return',
            'turnover','realized_vol_60','realized_vol_120']
    cols = [c for c in cols if c in r.columns]
    out = r[cols].copy()
    assert_rows(len(out), "Stage 5b features_daily")
    out.to_parquet(OUT, index=False)
    print(f"\
  -> written: {OUT.name}")
    print("=" * 78); print("STAGE 5b BUILD COMPLETE — run the diagnostic."); print("=" * 78)

  NOTE: 3 older-dated 'features_daily_v2' file(s) exist; reads use the newest, which will be features_daily_v2_2026-07-24.parquet after this build.
STAGE 5b — TURNOVER + REALIZED VOL
  turnover non-null: 3,259,324
  realized_vol_60   : 3,060,777 non-null
  realized_vol_120  : 2,908,952 non-null
  -> written: features_daily_v2_2026-07-24.parquet
STAGE 5b BUILD COMPLETE — run the diagnostic.


### Stage 5b — diagnostic *(gate)*

The warm-up assertion is the gate: `realized_vol_120` first appears on **exactly** trading day 121 for clean-start tickers, never earlier for any. Plus no-gap integrity (every vol value backed by a full real window), a hand-computed spot-check, and KSK's suspension yielding NaN (not fake ~0).

In [13]:
# =============================================================================
# STAGE 5b — DIAGNOSTIC (gate)   READ-ONLY
# =============================================================================
import numpy as np

f = pd.read_parquet(resolve('features_daily_v2')); f['Date'] = pd.to_datetime(f['Date'])
f = f.sort_values(['ticker','Date']).reset_index(drop=True)
W_S, W_L, ANN, DDOF = 60, 120, np.sqrt(252), 1

print("=" * 78); print("STAGE 5b DIAGNOSTIC"); print("=" * 78)
FAILURES = []
def check(name, cond, detail=""):
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}" + (f"  — {detail}" if detail else ""))
    if not cond: FAILURES.append(name)

print("\
[A] INTEGRITY")
check(f"row count == raw panel ({EXPECTED_ROWS:,})", len(f) == EXPECTED_ROWS)
for c in ['turnover','realized_vol_60','realized_vol_120']:
    check(f"{c} no inf", not np.isinf(f[c]).any())
    check(f"{c} no negatives", (f[c].dropna() >= 0).all())

print("\
[B] WARM-UP ASSERTION (vol first no earlier than window+1)")
def first_pos(s):
    idx = s.notna().values
    return int(np.argmax(idx)) + 1 if idx.any() else np.nan
v60, v120, cs60ok, cs60n, cs120ok, cs120n = [], [], 0, 0, 0, 0
for tkr, g in f.groupby('ticker'):
    g = g.reset_index(drop=True); dr = g['daily_return']
    p60 = first_pos(g['realized_vol_60'])
    if not np.isnan(p60):
        if p60 < W_S+1: v60.append(tkr)
        if dr.iloc[1:W_S+1].notna().all():
            cs60n += 1; cs60ok += (p60 == W_S+1)
    p120 = first_pos(g['realized_vol_120'])
    if not np.isnan(p120):
        if p120 < W_L+1: v120.append(tkr)
        if dr.iloc[1:W_L+1].notna().all():
            cs120n += 1; cs120ok += (p120 == W_L+1)
check("no vol_60 before day 61", len(v60) == 0)
check("no vol_120 before day 121", len(v120) == 0)
check("clean-start vol_60 exactly at day 61", cs60ok == cs60n, f"{cs60ok}/{cs60n}")
check("clean-start vol_120 exactly at day 121", cs120ok == cs120n, f"{cs120ok}/{cs120n}")

print("\
[C] NO-GAP INTEGRITY (300 sampled vol_120 backed by 120 real returns)")
np.random.seed(11)
samp = f[f['realized_vol_120'].notna()].sample(300, random_state=11)
gap_viol = 0
for row in samp.itertuples():
    g = f[f['ticker'] == row.ticker].sort_values('Date').reset_index(drop=True)
    pos = g.index[g['Date'] == row.Date][0]
    win = g['daily_return'].iloc[pos-W_L+1:pos+1]
    if len(win) != W_L or win.isna().any(): gap_viol += 1
check("300 sampled vol_120 each backed by 120 real returns", gap_viol == 0, f"{gap_viol} span a gap")

print("\
[D] HAND-COMPUTED SPOT-CHECK")
pick = f[(f['ticker']=='RELIANCE.NS') & f['realized_vol_120'].notna()].iloc[1000]
g = f[f['ticker']=='RELIANCE.NS'].sort_values('Date').reset_index(drop=True)
pos = g.index[g['Date']==pick['Date']][0]
manual = g['daily_return'].iloc[pos-W_L+1:pos+1].std(ddof=DDOF) * ANN
check("hand-computed vol_120 == stored", abs(manual - pick['realized_vol_120']) < 1e-9,
      f"diff {abs(manual-pick['realized_vol_120']):.2e}")

print("\
[E] COVERAGE BY YEAR")
f['_yr'] = f['Date'].dt.year
cov = f.groupby('_yr').agg(rows=('ticker','size'),
    turnover=('turnover', lambda s: 100*s.notna().mean()),
    vol60=('realized_vol_60', lambda s: 100*s.notna().mean()),
    vol120=('realized_vol_120', lambda s: 100*s.notna().mean()))
print(cov.to_string(float_format=lambda x: f"{x:,.2f}"))

print("\
[F] KSK.NS SUSPENSION — vol NaN, not ~0")
ksk = f[(f['ticker']=='KSK.NS') & (f['Date']>='2021-06-01') & (f['Date']<='2023-12-01')]
if len(ksk):
    check("KSK vol_120 NaN throughout suspension", ksk['realized_vol_120'].isna().all(),
          f"{ksk['realized_vol_120'].notna().sum()} non-null in gap")
else:
    print("    (KSK suspension window has no rows in this build — skipping)")

print("\
" + "=" * 78)
print("STAGE 5b DIAGNOSTIC: " + ("FAIL — " + ", ".join(FAILURES) if FAILURES else "PASS"))
print("=" * 78)

STAGE 5b DIAGNOSTIC
[A] INTEGRITY
  [PASS] row count == raw panel (3,328,132)
  [PASS] turnover no inf
  [PASS] turnover no negatives
  [PASS] realized_vol_60 no inf
  [PASS] realized_vol_60 no negatives
  [PASS] realized_vol_120 no inf
  [PASS] realized_vol_120 no negatives
[B] WARM-UP ASSERTION (vol first no earlier than window+1)
  [PASS] no vol_60 before day 61
  [PASS] no vol_120 before day 121
  [PASS] clean-start vol_60 exactly at day 61  — 981/981
  [PASS] clean-start vol_120 exactly at day 121  — 951/951
[C] NO-GAP INTEGRITY (300 sampled vol_120 backed by 120 real returns)
  [PASS] 300 sampled vol_120 each backed by 120 real returns  — 0 span a gap
[D] HAND-COMPUTED SPOT-CHECK
  [PASS] hand-computed vol_120 == stored  — diff 6.66e-16
[E] COVERAGE BY YEAR
        rows  turnover  vol60  vol120
_yr                                  
2010  178551     98.00  71.74   48.28
2011  182428     98.53  92.35   89.91
2012  186033     98.38  92.67   89.96
2013  189456     98.06  93.37   91.9

## Stage 5c — Monthly Amihud illiquidity  (build)

`daily_amihud = |daily_return| / turnover`; monthly mean per ticker. **Separate monthly file** (Q6 — never broadcast onto daily rows, which would leak future days). Zero-turnover days masked to NaN and excluded. A month needs ≥15 valid days; `n_valid_days` retained so the threshold is reversible.

In [14]:
# =============================================================================
# STAGE 5c — MONTHLY AMIHUD (build)   [requires Cell A]
# =============================================================================
import numpy as np

try:
    OUT = out_path('amihud_monthly_v2'); DO_BUILD = True
except FileExistsError as e:
    print(f"amihud_monthly_v2 exists today — skipping build. ({e})"); DO_BUILD = False

if DO_BUILD:
    MIN_DAYS = 15
    print("=" * 78); print("STAGE 5c — MONTHLY AMIHUD"); print("=" * 78)
    f = pd.read_parquet(resolve('features_daily_v2'),
                        columns=['Date','ticker','source','daily_return','turnover'])
    f['Date'] = pd.to_datetime(f['Date'])
    assert f['ticker'].nunique() == 1069, "wrong input. STOP."

    elig = f['daily_return'].notna() & f['turnover'].notna() & (f['turnover'] > 0)
    f['daily_amihud'] = np.where(elig, f['daily_return'].abs() / f['turnover'].where(f['turnover']>0), np.nan)
    assert not np.isinf(f['daily_amihud']).any(), "amihud inf — zero turnover survived. STOP."

    f['year_month'] = f['Date'].dt.to_period('M')
    f['_elig'] = np.isfinite(f['daily_amihud'])
    monthly = f.groupby(['ticker','year_month']).agg(
        trading_days=('Date','size'), n_valid_days=('_elig','sum'),
        amihud_raw=('daily_amihud','mean'), source=('source','first')).reset_index()
    monthly['amihud_illiquidity'] = monthly['amihud_raw'].where(monthly['n_valid_days'] >= MIN_DAYS)

    fm = monthly.groupby('ticker')['year_month'].transform('min')
    lm = monthly.groupby('ticker')['year_month'].transform('max')
    monthly['is_edge_month'] = (monthly['year_month']==fm) | (monthly['year_month']==lm)
    monthly['year_month'] = monthly['year_month'].astype(str)

    cols = ['ticker','year_month','trading_days','n_valid_days','amihud_illiquidity',
            'amihud_raw','is_edge_month','source']
    monthly[cols].to_parquet(OUT, index=False)
    n_pass = (monthly['n_valid_days'] >= MIN_DAYS).sum()
    print(f"  stock-months: {len(monthly):,}   >=15 days: {n_pass:,}   "
          f"amihud non-null: {monthly['amihud_illiquidity'].notna().sum():,}")
    print(f"\
  -> written: {OUT.name}")
    print("=" * 78); print("STAGE 5c BUILD COMPLETE — run the diagnostic."); print("=" * 78)

  NOTE: 3 older-dated 'amihud_monthly_v2' file(s) exist; reads use the newest, which will be amihud_monthly_v2_2026-07-24.parquet after this build.
STAGE 5c — MONTHLY AMIHUD
  stock-months: 162,783   >=15 days: 157,930   amihud non-null: 157,930
  -> written: amihud_monthly_v2_2026-07-24.parquet
STAGE 5c BUILD COMPLETE — run the diagnostic.


### Stage 5c — diagnostic *(gate)*

No inf, threshold applied correctly, a hand-computed month, the no-leakage structure (monthly grain, no Date column), and the distribution sanity check (right-skewed; large caps most liquid, dead micro-caps most illiquid).

In [15]:
# =============================================================================
# STAGE 5c — DIAGNOSTIC (gate)   READ-ONLY
# =============================================================================
import numpy as np

a = pd.read_parquet(resolve('amihud_monthly_v2'))
MIN_DAYS = 15
print("=" * 78); print("STAGE 5c DIAGNOSTIC"); print("=" * 78)
FAILURES = []
def check(name, cond, detail=""):
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}" + (f"  — {detail}" if detail else ""))
    if not cond: FAILURES.append(name)

print("\
[A] INTEGRITY")
check("no duplicate (ticker, year_month)", not a.duplicated(['ticker','year_month']).any())
check("amihud no inf", not np.isinf(a['amihud_illiquidity'].dropna()).any())
check("amihud no negatives", (a['amihud_illiquidity'].dropna() >= 0).all())
check("every non-null amihud has >=15 valid days", (a.loc[a['amihud_illiquidity'].notna(),'n_valid_days'] >= MIN_DAYS).all())
check("every month <15 days has NaN amihud", a.loc[a['n_valid_days'] < MIN_DAYS, 'amihud_illiquidity'].isna().all())
print(f"    stock-months: {len(a):,}   amihud non-null: {a['amihud_illiquidity'].notna().sum():,}")

print("\
[B] HAND-COMPUTED SPOT-CHECK")
f = pd.read_parquet(resolve('features_daily_v2'), columns=['Date','ticker','daily_return','turnover'])
f['Date'] = pd.to_datetime(f['Date']); f['year_month'] = f['Date'].dt.to_period('M').astype(str)
pick = a[(a['ticker']=='RELIANCE.NS') & a['amihud_illiquidity'].notna()].iloc[100]
sub = f[(f['ticker']=='RELIANCE.NS') & (f['year_month']==pick['year_month'])]
elig = sub[(sub['daily_return'].notna()) & (sub['turnover']>0)]
manual = (elig['daily_return'].abs()/elig['turnover']).mean()
check("hand-computed monthly Amihud == stored", np.isclose(manual, pick['amihud_illiquidity'], rtol=1e-9))
check("n_valid_days == manual eligible count", pick['n_valid_days'] == len(elig), f"{pick['n_valid_days']} vs {len(elig)}")

print("\
[C] NO-LEAKAGE STRUCTURE (Q6)")
check("one row per (ticker, year_month)", not a.duplicated(['ticker','year_month']).any())
check("no Date column (not broadcast onto daily rows)", 'Date' not in a.columns)

print("\
[D] DISTRIBUTION SANITY (right-skewed)")
v = a['amihud_illiquidity'].dropna()
check("mean > median (right skew)", v.mean() > v.median(), f"ratio {v.mean()/v.median():.1f}")
print("    most LIQUID (lowest amihud) — expect large caps:")
print(a[a['amihud_illiquidity'].notna()].nsmallest(5,'amihud_illiquidity')
      [['ticker','year_month','n_valid_days','amihud_illiquidity']].to_string(index=False, float_format=lambda x: f"{x:.3e}"))

print("\
" + "=" * 78)
print("STAGE 5c DIAGNOSTIC: " + ("FAIL — " + ", ".join(FAILURES) if FAILURES else "PASS"))
print("=" * 78)

STAGE 5c DIAGNOSTIC
[A] INTEGRITY
  [PASS] no duplicate (ticker, year_month)
  [PASS] amihud no inf
  [PASS] amihud no negatives
  [PASS] every non-null amihud has >=15 valid days
  [PASS] every month <15 days has NaN amihud
    stock-months: 162,783   amihud non-null: 157,930
[B] HAND-COMPUTED SPOT-CHECK
  [PASS] hand-computed monthly Amihud == stored
  [PASS] n_valid_days == manual eligible count  — 22 vs 22
[C] NO-LEAKAGE STRUCTURE (Q6)
  [PASS] one row per (ticker, year_month)
  [PASS] no Date column (not broadcast onto daily rows)
[D] DISTRIBUTION SANITY (right-skewed)
  [PASS] mean > median (right skew)  — ratio 681.9
    most LIQUID (lowest amihud) — expect large caps:
     ticker year_month  n_valid_days  amihud_illiquidity
 VIVIDHA.NS    2020-05            15           0.000e+00
HDFCBANK.NS    2023-09            20           2.069e-13
HDFCBANK.NS    2023-08            22           2.190e-13
HDFCBANK.NS    2024-03            18           2.271e-13
HDFCBANK.NS    2025-09        

## Stage 6 — Join PIT flag as metadata  (build)

Left-joins the daily PIT flag onto the feature panel on `(Date, ticker)`. Membership is metadata — nothing is filtered. The join must preserve row count exactly (a change means duplicate keys) and leave every feature value bit-identical.

In [16]:
# =============================================================================
# STAGE 6 — JOIN PIT FLAG AS METADATA (build)   [requires Cell A]
# =============================================================================
import numpy as np

try:
    OUT = out_path('feature_panel_with_pit_v2'); DO_BUILD = True
except FileExistsError as e:
    print(f"feature_panel_with_pit_v2 exists today — skipping build. ({e})"); DO_BUILD = False

if DO_BUILD:
    print("=" * 78); print("STAGE 6 — JOIN PIT FLAG AS METADATA"); print("=" * 78)
    feat = pd.read_parquet(resolve('features_daily_v2')); feat['Date'] = pd.to_datetime(feat['Date'])
    flag = pd.read_parquet(resolve('daily_pit_flag')); flag['Date'] = pd.to_datetime(flag['Date'])
    FEAT_ROWS = len(feat)
    assert feat['ticker'].nunique() == 1069, "wrong features. STOP."

    assert flag.duplicated(['Date','ticker']).sum() == 0, "Flag has dup keys. STOP."
    assert feat.duplicated(['Date','ticker']).sum() == 0, "Features have dup keys. STOP."

    # The PIT flag was built over the FULL v1/v2 date range, which still contains
    # the market-closed dates that v3 removed. Restrict the flag to the dates the
    # feature panel actually has, so the key-set equality assert below tests what
    # it is meant to test (flag covers every feature row) rather than failing on
    # dates that no longer exist by design.
    _feat_dates = set(feat['Date'].unique())
    _flag_before = len(flag)
    flag = flag[flag['Date'].isin(_feat_dates)].reset_index(drop=True)
    if len(flag) != _flag_before:
        print(f"  flag restricted to v3 dates: {_flag_before:,} -> {len(flag):,} rows "
              f"({_flag_before - len(flag):,} dropped on removed market-closed dates)")
    feat_keys = set(map(tuple, feat[['Date','ticker']].values))
    flag_keys = set(map(tuple, flag[['Date','ticker']].values))
    assert len(feat_keys - flag_keys) == 0, "Feature rows without a flag. STOP."
    assert len(flag_keys - feat_keys) == 0, "Flag rows without a feature. STOP."
    print("  key sets identical — flag covers the panel exactly")

    fcols = [c for c in ['Date','ticker','in_nifty500_flag','flag_is_extrapolated',
                         'snapshot_source','effective_snap_date'] if c in flag.columns]
    overlap = (set(feat.columns) & set(fcols)) - {'Date','ticker'}
    assert len(overlap) == 0, f"Column collision {overlap}. STOP."

    fvcols = ['overnight_return','intraday_return','daily_return','turnover','realized_vol_60','realized_vol_120']
    before = feat[['Date','ticker']+fvcols].copy()
    panel = feat.merge(flag[fcols], on=['Date','ticker'], how='left', validate='one_to_one')
    assert len(panel) == FEAT_ROWS, "Row count changed — fan-out. STOP."

    ps = panel.sort_values(['ticker','Date']).reset_index(drop=True)
    bs = before.sort_values(['ticker','Date']).reset_index(drop=True)
    for c in fvcols:
        assert bs[c].equals(ps[c]), f"{c} changed during join. STOP."
    assert (panel['in_nifty500_flag']==1).sum() == (flag['in_nifty500_flag']==1).sum(), "flag==1 changed. STOP."
    assert panel['in_nifty500_flag'].isna().sum() == flag['in_nifty500_flag'].isna().sum(), "flag NaN changed. STOP."

    assert_rows(len(panel), "Stage 6 feature_panel_with_pit")
    panel.to_parquet(OUT, index=False)
    print(f"  flag==1: {(panel['in_nifty500_flag']==1).sum():,}  "
          f"flag==0: {(panel['in_nifty500_flag']==0).sum():,}  "
          f"NaN: {panel['in_nifty500_flag'].isna().sum():,}")
    print(f"\
  -> written: {OUT.name}")
    print("=" * 78); print("STAGE 6 JOIN COMPLETE — run the diagnostic."); print("=" * 78)

  NOTE: 3 older-dated 'feature_panel_with_pit_v2' file(s) exist; reads use the newest, which will be feature_panel_with_pit_v2_2026-07-24.parquet after this build.
STAGE 6 — JOIN PIT FLAG AS METADATA
  flag restricted to v3 dates: 3,332,879 -> 3,328,132 rows (4,747 dropped on removed market-closed dates)
  key sets identical — flag covers the panel exactly
  flag==1: 1,973,146  flag==0: 1,269,522  NaN: 85,464
  -> written: feature_panel_with_pit_v2_2026-07-24.parquet
STAGE 6 JOIN COMPLETE — run the diagnostic.


### Stage 6 — diagnostic *(gate)*

Row count and features bit-identical to pre-join; flag distribution matches the standalone flag; tri-state boundary intact; HDFC/RANBAXY survive; metadata populated.

In [25]:
# =============================================================================
# STAGE 6 — DIAGNOSTIC (gate)   READ-ONLY
# =============================================================================
import numpy as np

p = pd.read_parquet(resolve('feature_panel_with_pit_v2')); p['Date'] = pd.to_datetime(p['Date'])
feat = pd.read_parquet(resolve('features_daily_v2')); feat['Date'] = pd.to_datetime(feat['Date'])
flag = pd.read_parquet(resolve('daily_pit_flag')); flag['Date'] = pd.to_datetime(flag['Date'])

print("=" * 78); print("STAGE 6 DIAGNOSTIC"); print("=" * 78)
FAILURES = []
def check(name, cond, detail=""):
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}" + (f"  — {detail}" if detail else ""))
    if not cond: FAILURES.append(name)

print("\n[A] INTEGRITY")
if EXPECTED_ROWS is None:
    print("  [SKIP] row count — EXPECTED_ROWS unset. Re-run Cell A.")
else:
    check(f"row count == raw panel ({EXPECTED_ROWS:,})", len(p) == EXPECTED_ROWS)
check("row count == pre-join", len(p) == len(feat))
check("no duplicate (Date,ticker)", not p.duplicated(['Date','ticker']).any())

print("\n[B] FEATURES BIT-IDENTICAL TO PRE-JOIN")
ps = p.sort_values(['ticker','Date']).reset_index(drop=True)
fs = feat.sort_values(['ticker','Date']).reset_index(drop=True)
for c in ['overnight_return','intraday_return','daily_return','turnover','realized_vol_60','realized_vol_120']:
    check(f"{c} unchanged", fs[c].equals(ps[c]))

print("\n[C] FLAG DISTRIBUTION MATCHES STANDALONE")
# The daily_pit_flag file was built over the FULL v1/v2 date range and still
# contains rows on the six market-closed dates that v3 removed. Comparing the
# panel against the UNRESTRICTED flag fails by construction — the panel
# legitimately has fewer rows. Restrict the flag to the panel's dates first.
_panel_dates = set(p['Date'].unique())
flag_v3 = flag[flag['Date'].isin(_panel_dates)]
_dropped = len(flag) - len(flag_v3)
print(f"    flag restricted to panel dates: {len(flag):,} -> {len(flag_v3):,} "
      f"({_dropped:,} rows on removed market-closed dates)")
for label, val in [("flag==1",1),("flag==0",0)]:
    check(f"{label} matches", (p['in_nifty500_flag']==val).sum() == (flag_v3['in_nifty500_flag']==val).sum())
check("flag NaN matches", p['in_nifty500_flag'].isna().sum() == flag_v3['in_nifty500_flag'].isna().sum())

print("\n[D] TRI-STATE BOUNDARY INTACT")
check("no NaN flag on/after 2010-06-30", len(p[p['in_nifty500_flag'].isna() & (p['Date'] >= pd.Timestamp('2010-06-30'))]) == 0)

print("\n[E] REAL MEMBERSHIP SURVIVES")
h = p[(p['ticker']=='HDFC.NS') & (p['in_nifty500_flag']==1)]
if len(h): check("HDFC last flag==1 on/before 2023-07-12", h['Date'].max() <= pd.Timestamp('2023-07-12'))

print("\n[F] METADATA POPULATED")
check("snapshot_source present on all non-NaN-flag rows",
      p[p['in_nifty500_flag'].notna()]['snapshot_source'].isna().sum() == 0)
carry = p[p['snapshot_source'].astype(str).str.contains('carried_forward', na=False)]
print(f"    2018 carry-forward rows tagged: {len(carry):,}")

print("\n" + "=" * 78)
print("STAGE 6 DIAGNOSTIC: " + ("FAIL — " + ", ".join(FAILURES) if FAILURES else "PASS"))
print("=" * 78)

STAGE 6 DIAGNOSTIC

[A] INTEGRITY
  [PASS] row count == raw panel (3,328,132)
  [PASS] row count == pre-join
  [PASS] no duplicate (Date,ticker)

[B] FEATURES BIT-IDENTICAL TO PRE-JOIN
  [PASS] overnight_return unchanged
  [PASS] intraday_return unchanged
  [PASS] daily_return unchanged
  [PASS] turnover unchanged
  [PASS] realized_vol_60 unchanged
  [PASS] realized_vol_120 unchanged

[C] FLAG DISTRIBUTION MATCHES STANDALONE
    flag restricted to panel dates: 3,332,879 -> 3,328,132 (4,747 rows on removed market-closed dates)
  [PASS] flag==1 matches
  [PASS] flag==0 matches
  [PASS] flag NaN matches

[D] TRI-STATE BOUNDARY INTACT
  [PASS] no NaN flag on/after 2010-06-30

[E] REAL MEMBERSHIP SURVIVES
  [PASS] HDFC last flag==1 on/before 2023-07-12

[F] METADATA POPULATED
  [PASS] snapshot_source present on all non-NaN-flag rows
    2018 carry-forward rows tagged: 99,759

STAGE 6 DIAGNOSTIC: PASS


In [24]:
flag_full = pd.read_parquet(resolve('daily_pit_flag'))
flag_full['Date'] = pd.to_datetime(flag_full['Date'])
p = pd.read_parquet(resolve('feature_panel_with_pit_v2'))
p['Date'] = pd.to_datetime(p['Date'])

REMOVED = [pd.Timestamp(d) for d in
           ['2014-04-24','2014-10-15','2026-01-15','2026-05-01','2026-05-28','2026-06-26']]
flag_v3 = flag_full[~flag_full['Date'].isin(REMOVED)]

for lbl, v in [("flag==1", 1), ("flag==0", 0)]:
    a = int((p['in_nifty500_flag'] == v).sum())
    b = int((flag_v3['in_nifty500_flag'] == v).sum())
    c = int((flag_full['in_nifty500_flag'] == v).sum())
    print(f"{lbl}: panel={a:,}  flag_restricted={b:,}  flag_full={c:,}  "
          f"match_restricted={a==b}  diff_vs_full={c-a:,}")

print("\nflag rows on removed dates:", int(flag_full['Date'].isin(REMOVED).sum()))

flag==1: panel=1,973,146  flag_restricted=1,973,146  flag_full=1,975,928  match_restricted=True  diff_vs_full=2,782
flag==0: panel=1,269,522  flag_restricted=1,269,522  flag_full=1,271,487  match_restricted=True  diff_vs_full=1,965

flag rows on removed dates: 4747


## Manifest — build `manifest.yaml`

Machine-readable ledger: every artifact with its measured row count, full lineage, ratified decisions, and every logged limitation/deferred item. Measured from the files themselves, not from memory.

In [19]:
 # #############################################################################
# CELL 13 — BUILD manifest.yaml   *** THIS WAS MISSING FROM THE NOTEBOOK ***
#
# The notebook jumped straight from the Stage 6 join diagnostic (Cell 12) to
# the manifest READ-BACK (Cell 14). The cell that WRITES the manifest was never
# pasted in, so Cell 14 tried to open a file nothing had created -> the
# FileNotFoundError you hit.
#
# This cell writes it. It uses the resolver from Cell A, so it finds the newest
# of each artifact regardless of what date they were built on. Runs on any day.
#
# Requires Cell A (config + resolver) to have run first.
# #############################################################################

import pandas as pd
import numpy as np
import yaml

print("=" * 90)
print("CELL 13 — BUILD manifest.yaml")
print("=" * 90)

MANIFEST = out_path('manifest')   # refuses to overwrite an existing manifest

# --- Measure each artifact from the file itself (never from memory) ---------
def measure(key, monthly=False):
    path = resolve(key)
    df = pd.read_parquet(path)
    info = {'filename': path.name, 'rows': int(len(df)),
            'size_mb': round(path.stat().st_size / 1e6, 2)}
    if 'ticker' in df.columns:
        info['tickers'] = int(df['ticker'].nunique())
    if 'Date' in df.columns:
        info['date_min'] = str(pd.to_datetime(df['Date']).min().date())
        info['date_max'] = str(pd.to_datetime(df['Date']).max().date())
    if monthly and 'year_month' in df.columns:
        info['month_min'] = str(df['year_month'].min())
        info['month_max'] = str(df['year_month'].max())
    if 'price_status' in df.columns:
        info['price_status_counts'] = {k: int(v) for k, v in df['price_status'].value_counts().items()}
    if 'in_nifty500_flag' in df.columns:
        fl = df['in_nifty500_flag']
        info['flag_counts'] = {'in_index': int((fl == 1).sum()),
                               'not_in_index': int((fl == 0).sum()),
                               'unknown_nan': int(fl.isna().sum())}
    if 'amihud_illiquidity' in df.columns:
        info['amihud_non_null'] = int(df['amihud_illiquidity'].notna().sum())
    for c in ['realized_vol_60', 'realized_vol_120', 'turnover',
              'overnight_return', 'intraday_return', 'daily_return']:
        if c in df.columns:
            info.setdefault('feature_non_null', {})[c] = int(df[c].notna().sum())
    return info

print("\n[1] MEASURING ARTIFACTS")
artifacts = {}
for key in ['price_panel_v1', 'price_panel_v3', 'daily_pit_flag',
            'returns_panel_v2', 'features_daily_v2', 'amihud_monthly_v2',
            'feature_panel_with_pit_v2']:
    path = resolve(key, required=(key != 'price_panel_v1'))
    if path is None:
        print(f"  {key:<28}: (not present — skipping)")
        continue
    artifacts[key] = measure(key, monthly=(key == 'amihud_monthly_v2'))
    print(f"  {key:<28}: {artifacts[key]['rows']:>10,} rows")

# --- Assemble ---------------------------------------------------------------
print("\n[2] ASSEMBLING MANIFEST")
manifest = {
    'project': 'moonshot_compounding',
    'manifest_version': 1,
    'manifest_build_date': BUILD_DATE,
    'description': (
        'Point-in-time daily feature panel for the NSE Nifty 500 universe '
        '(2010-present), plus a separate monthly Amihud illiquidity file. '
        'Shared foundation for six planned hypothesis tests.'
    ),
    'lineage': {
        'raw_price_panel_v1': {
            'description': 'Stage 3 output. 955 yfinance + 114 Bloomberg tickers.',
            'status': 'SUPERSEDED by v2. Retained, never modified.'},
        'raw_price_panel_v3': {
            'description': 'Stage 3B remediation of v1. Three defect classes masked to NaN.',
            'derived_from': 'raw_price_panel_v1',
            'status': 'CURRENT raw panel.',
            'changes': {'phantom_rows_masked': 57876, 'split_splices_masked': 8,
                        'bad_prints_masked': 6, 'total_rows_masked': 57890,
                        'action': 'OHLC set to NaN; Volume and row retained; price_status added',
                        'bloomberg_rows_touched': 0}},
        'daily_pit_flag': {
            'description': 'Stage 4. Daily NIFTY 500 membership via explicit as_of join.',
            'derived_from': '33 semi-annual snapshots + pull_ticker_list_v1.csv (ISIN join)',
            'status': 'CURRENT. Not rebuilt from v2 — flag never touches a price; '
                      'v2 (Date,ticker) keys are bit-identical to v1.'},
        'returns_panel_v2': {'description': 'Stage 5a. overnight, intraday, daily returns.',
                             'derived_from': 'raw_price_panel_v3'},
        'features_daily_v2': {'description': 'Stage 5b. Adds turnover, realized_vol_60/120.',
                              'derived_from': 'returns_panel_v2'},
        'amihud_monthly_v2': {'description': 'Stage 5c. Monthly Amihud. SEPARATE file (Q6, no leakage).',
                              'derived_from': 'features_daily_v2'},
        'feature_panel_with_pit_v2': {
            'description': 'Stage 6. THE DELIVERABLE. Daily features + PIT flag as metadata.',
            'derived_from': 'features_daily_v2 LEFT JOIN daily_pit_flag on (Date,ticker)',
            'status': 'CURRENT deliverable.'},
    },
    'artifacts': artifacts,
    'features': {
        'overnight_return': 'open[t]/close[t-1] - 1, prior TRADING row',
        'intraday_return': 'close[t]/open[t] - 1',
        'daily_return': 'close[t]/close[t-1] - 1',
        'turnover': 'volume[t] * close[t]',
        'realized_vol_60': 'rolling std daily_return, window 60, min_periods=60, ddof=1, x sqrt(252)',
        'realized_vol_120': 'rolling std daily_return, window 120, min_periods=120, ddof=1, x sqrt(252)',
        'amihud_illiquidity': 'monthly mean of |daily_return|/turnover; >=15 valid days; SEPARATE monthly file',
    },
    'decisions': {
        'Q1_pit_join_key': 'ISIN via pull_ticker_list_v1.csv. Every ISIN -> one exch_symbol across all 33 snapshots.',
        'Q2_forward_fill_tail': 'Hold 2026-06-01 flag flat; tag flag_is_extrapolated=True. Refresh early April / early October.',
        'Q3_leading_edge': 'flag=NaN before 2010-06-30. Tri-state. Never fabricate 0; never backfill (survivorship).',
        'Q4_ohlc_adjustment': 'Open/High/Low/Close all split-adjusted on same scale (582 events). No adjustment layer.',
        'Q6_amihud_shape': 'Separate monthly file. NOT broadcast onto daily rows (would leak future days).',
        'Q7_zero_turnover': 'turnover==0 masked to NaN, counted, excluded from Amihud mean.',
        'Q8_rolling_windows': 'min_periods==window (no partial). ddof=1.',
        'Q9_annualization': 'sqrt(252). Constant scalar; cannot affect any cross-sectional sort.',
        'Q10_ticker_count': '1,069 (955 yfinance + 114 Bloomberg). MIRCELECTR.NS, DENABANK.NS absent (permanent gaps).',
        'remediation_principle': 'When the price is unknown, say so (NaN). Never substitute a plausible number. No interpolation/forward-fill/winsorize/deletion.',
        'split_splice_handling': 'NaN the 8 boundary rows; do NOT rescale pre-history (memory fact #1).',
    },
    'known_limitations': {
        'semi_annual_pit_resolution': (
            'The PIT flag is point-in-time AS OF THE LAST SEMI-ANNUAL SNAPSHOT, not the actual '
            'index event. Membership can be off up to ~6 months either way. RANBAXY.NS flag drops '
            '2014-12-31 while the stock traded to 2015-04-01.'),
        'window_2018_carry_forward': (
            '2018-07-01 to 2018-12-30 membership carried forward from 2017-12-31 (2018-06-30 '
            'snapshot truncated). ~22% potential misclassification. 99,759 rows tagged '
            'snapshot_source="carried_forward_from_2017-12-31".'),
        'constituents_without_prices': (
            '5 ISINs are genuine constituents in recent snapshots but have no price data (Vedanta '
            'x3, Malco Energy, Kwality Wall). Recent-window universe ~495 flagged vs ~500 true. '
            'See validation/pit_constituents_without_prices_*.csv'),
        'split_price_levels_incomparable': (
            'For the 8 splice tickers, price LEVELS are not comparable across the split boundary. '
            'RETURNS within each window are correct (scale-invariant).'),
        'thin_vol_coverage_years': (
            'realized_vol_120 coverage dips in 2014 (~36%) and 2025 (~49%) beyond pure warm-up, '
            'from universe churn and new-listing/masked-run clustering. Treat concentrated results '
            'in these years with extra caution.'),
    },
    'deferred_to_anomaly_review': {
        'stale_price_runs': (
            'Tickers quoting a stuck price on real volume, or long suspension runs. KENNAMET.NS '
            '861-day gap. KSK.NS ~30-month suspension (now NaN vol, not fake ~0). VIVIDHA.NS 2020-05 '
            'Amihud exactly 0 (flat-but-traded). Needs a consecutive-identical-close rule.'),
        'vstind_2024_09_06': (
            '+17% overnight on an 11x split date. Passed Q4 ADJUSTED comfortably but large for a '
            'split day. Possibly a bonus-plus-split the ratio does not capture.'),
    },
    'methodological_note': (
        'THRESHOLDS CATCH LOUD ERRORS; ONLY STRUCTURE CATCHES QUIET ONES. The +/-30% review list '
        'caught 193 of ~21,472 phantom-boundary artifacts (0.9%). Every Stage 3 validation was '
        'threshold-based and the contamination was sub-threshold by construction. What found it was '
        'searching for a known fabrication signature (O==H==L==C AND Volume==0), not for large numbers.'),
    'data_sources': {
        'yfinance': '955 tickers. Close already split/bonus-adjusted. No-trade encoded as Volume=0. Fabricates phantom rows.',
        'bloomberg': '114 tickers. Excel-Add-in only. No-trade encoded as Volume=NaN. Zero phantom rows.',
    },
    'maintenance': {
        'refresh_schedule': 'Append a new NIFTY 500 snapshot early April and early October (just after each review). Never January.',
        'refresh_effect': 'On refresh, flag_is_extrapolated rows in the un-refreshed tail flip to observed.',
        'never_overwrite': 'All files timestamped, never overwritten. Build cells refuse to overwrite same-day files.',
    },
}

print("\n[3] WRITING manifest.yaml")
with open(MANIFEST, 'w') as fh:
    yaml.safe_dump(manifest, fh, sort_keys=False, default_flow_style=False, width=100)
print(f"  -> written: {MANIFEST}")
print(f"     {MANIFEST.stat().st_size/1024:.1f} KB")
print("\n" + "=" * 90)
print("MANIFEST BUILD COMPLETE — run Cell 14 (read-back) to verify.")
print("=" * 90)

CELL 13 — BUILD manifest.yaml


FileExistsError: manifest.yaml already exists (built today). This does not overwrite. To RE-READ it, use resolve('manifest') in the diagnostic. To rebuild, delete the file or pass overwrite=True deliberately.

### Manifest — read-back *(verify)*

Re-opens the YAML, confirms it parses, and cross-checks recorded row counts against the actual files.

In [20]:
# =============================================================================
# MANIFEST READ-BACK (verify)   READ-ONLY
# =============================================================================
import yaml

m = yaml.safe_load(resolve('manifest').read_text())
print("=" * 78); print("MANIFEST READ-BACK"); print("=" * 78)
FAILURES = []
def check(name, cond, detail=""):
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}" + (f"  — {detail}" if detail else ""))
    if not cond: FAILURES.append(name)

check("parses as dict", isinstance(m, dict))
print(f"  top-level keys: {list(m.keys())}")

print("\
[recorded counts vs actual files]")
for key in ['price_panel_v2','daily_pit_flag','features_daily_v2','amihud_monthly_v2','feature_panel_with_pit_v2']:
    if key not in m['artifacts']:
        check(f"{key} in manifest", False); continue
    recorded = m['artifacts'][key]['rows']
    actual = len(pd.read_parquet(resolve(key)))
    check(f"{key} row count matches", recorded == actual, f"manifest {recorded:,} vs file {actual:,}")

d = m['artifacts'].get('feature_panel_with_pit_v2', {})
if d:
    print(f"\
  DELIVERABLE: {d['filename']}")
    print(f"    rows {d['rows']:,}  tickers {d.get('tickers')}  dates {d.get('date_min')} -> {d.get('date_max')}")
    print(f"    flag {d.get('flag_counts')}")

print("\
" + "=" * 78)
if FAILURES:
    print("MANIFEST READ-BACK: FAIL — " + ", ".join(FAILURES))
else:
    print("MANIFEST READ-BACK: PASS")
    print("\
  Stages 4-6 complete. All seven features built, PIT flag joined as")
    print("  metadata, every exception logged, lineage recorded, manifest verified.")
    print("  Ready for hypothesis test development.")
print("=" * 78)

MANIFEST READ-BACK
  [PASS] parses as dict
  top-level keys: ['project', 'manifest_version', 'manifest_build_date', 'description', 'lineage', 'artifacts', 'features', 'decisions', 'known_limitations', 'deferred_to_anomaly_review', 'methodological_note', 'data_sources', 'maintenance']
[recorded counts vs actual files]
  [PASS] price_panel_v2 row count matches  — manifest 3,332,879 vs file 3,332,879
  [PASS] daily_pit_flag row count matches  — manifest 3,332,879 vs file 3,332,879
  [FAIL] features_daily_v2 row count matches  — manifest 3,332,879 vs file 3,328,132
  [PASS] amihud_monthly_v2 row count matches  — manifest 162,783 vs file 162,783
  [FAIL] feature_panel_with_pit_v2 row count matches  — manifest 3,332,879 vs file 3,328,132
  DELIVERABLE: feature_panel_with_pit_v2_2026-07-20.parquet
    rows 3,332,879  tickers 1069  dates 2010-01-04 -> 2026-07-10
    flag {'in_index': 1975928, 'not_in_index': 1271487, 'unknown_nan': 85464}
MANIFEST READ-BACK: FAIL — features_daily_v2 row count 

---

## Done

The deliverable is `feature_panel_with_pit_v2_*.parquet` (daily: 6 features + PIT metadata) plus `amihud_monthly_v2_*.parquet` (monthly Amihud, kept separate to avoid leakage). Everything is reproducible from `manifest.yaml`.

**To re-run later:** mount → Cell A → run down. Finished stages skip their build (refuse same-day overwrite) and their diagnostics re-verify the newest files. To rebuild a stage, delete its output file first.

In [21]:
# =============================================================================
# DUPLICATE-FILE CHECK — read-only. Lists ALL dated versions of each artifact.
# The resolver uses newest-wins, so an older copy doesn't break anything — but
# it's clutter and a future foot-gun. This surfaces any that exist.
# Requires Cell A (for BASE and REGISTRY) to have run.
# =============================================================================
from pathlib import Path

print("=" * 78)
print("DUPLICATE / STRAY FILE CHECK")
print("=" * 78)

# Every output artifact that carries a date stamp
dated_keys = ['price_panel_v2', 'daily_pit_flag', 'returns_panel_v2',
              'features_daily_v2', 'amihud_monthly_v2', 'feature_panel_with_pit_v2']

any_dupes = False
for key in dated_keys:
    spec = REGISTRY[key]
    folder = BASE if spec['folder'] == '.' else BASE / spec['folder']
    hits = sorted(folder.glob(spec['pattern']))
    marker = "" if len(hits) <= 1 else "  <-- MULTIPLE VERSIONS"
    if len(hits) > 1:
        any_dupes = True
    print(f"\n{key}  ({len(hits)} file{'s' if len(hits)!=1 else ''}){marker}")
    for i, p in enumerate(hits):
        newest = "  [NEWEST — resolver uses this]" if i == len(hits) - 1 else "  [older — superseded]"
        print(f"    {p.name}   {p.stat().st_size/1e6:,.1f} MB{newest}")

# Also check the validation/ folder for accumulated CSVs (harmless, but noisy)
print(f"\n{'-'*78}")
print("validation/ folder (CSVs — harmless, listed for awareness):")
vfiles = sorted((BASE / 'validation').glob('*.csv'))
for p in vfiles:
    print(f"    {p.name}")

print("\n" + "=" * 78)
if any_dupes:
    print("ACTION: some artifacts have multiple dated versions. The resolver uses")
    print("the newest, so nothing is broken — but you may want to delete the older")
    print("copies for a single canonical file each. Delete manually and deliberately.")
else:
    print("CLEAN: exactly one version of each artifact. Nothing to remove.")
print("=" * 78)

DUPLICATE / STRAY FILE CHECK

price_panel_v2  (3 files)  <-- MULTIPLE VERSIONS
    raw_price_panel_consolidated_v2_2026-07-17.parquet   93.7 MB  [older — superseded]
    raw_price_panel_consolidated_v2_2026-07-21.parquet   93.7 MB  [older — superseded]
    raw_price_panel_consolidated_v2_2026-07-22.parquet   93.7 MB  [NEWEST — resolver uses this]

daily_pit_flag  (6 files)  <-- MULTIPLE VERSIONS
    daily_pit_flag_v1_2026-07-16.parquet   5.3 MB  [older — superseded]
    daily_pit_flag_v1_2026-07-18.parquet   5.3 MB  [older — superseded]
    daily_pit_flag_v1_2026-07-20.parquet   5.3 MB  [older — superseded]
    daily_pit_flag_v1_2026-07-21.parquet   5.3 MB  [older — superseded]
    daily_pit_flag_v1_2026-07-22.parquet   5.3 MB  [older — superseded]
    daily_pit_flag_v1_2026-07-24.parquet   5.3 MB  [NEWEST — resolver uses this]

returns_panel_v2  (5 files)  <-- MULTIPLE VERSIONS
    returns_panel_v2_2026-07-18.parquet   175.1 MB  [older — superseded]
    returns_panel_v2_2026-07-20.par

In [22]:
# =============================================================================
# VERIFY DUPLICATES ARE IDENTICAL before deleting anything. Read-only.
# =============================================================================
import hashlib
from pathlib import Path

def sha(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p,'rb') as f:
        while (b := f.read(chunk)): h.update(b)
    return h.hexdigest()[:16]

groups = {
    'price_panel_v2': ('raw/prices','raw_price_panel_consolidated_v2_*.parquet'),
    'daily_pit_flag': ('universe','daily_pit_flag_v1_*.parquet'),
    'returns_panel_v2': ('features','returns_panel_v2_*.parquet'),
    'features_daily_v2': ('features','features_daily_v2_*.parquet'),
    'amihud_monthly_v2': ('features','amihud_monthly_v2_*.parquet'),
    'feature_panel_with_pit_v2': ('features','feature_panel_with_pit_v2_*.parquet'),
}
print("Checking whether duplicate versions are byte-identical:\n")
for key,(folder,pat) in groups.items():
    hits = sorted((BASE/folder).glob(pat))
    hashes = {p.name: sha(p) for p in hits}
    uniq = set(hashes.values())
    status = "ALL IDENTICAL" if len(uniq)==1 else f"*** {len(uniq)} DISTINCT VERSIONS ***"
    print(f"{key}: {status}")
    for n,h in hashes.items():
        print(f"    {h}  {n}")
    print()

Checking whether duplicate versions are byte-identical:

price_panel_v2: ALL IDENTICAL
    6939ab810ca288e2  raw_price_panel_consolidated_v2_2026-07-17.parquet
    6939ab810ca288e2  raw_price_panel_consolidated_v2_2026-07-21.parquet
    6939ab810ca288e2  raw_price_panel_consolidated_v2_2026-07-22.parquet

daily_pit_flag: ALL IDENTICAL
    aa09ea22606c1411  daily_pit_flag_v1_2026-07-16.parquet
    aa09ea22606c1411  daily_pit_flag_v1_2026-07-18.parquet
    aa09ea22606c1411  daily_pit_flag_v1_2026-07-20.parquet
    aa09ea22606c1411  daily_pit_flag_v1_2026-07-21.parquet
    aa09ea22606c1411  daily_pit_flag_v1_2026-07-22.parquet
    aa09ea22606c1411  daily_pit_flag_v1_2026-07-24.parquet

returns_panel_v2: *** 2 DISTINCT VERSIONS ***
    4241c7a4cc203539  returns_panel_v2_2026-07-18.parquet
    4241c7a4cc203539  returns_panel_v2_2026-07-20.parquet
    4241c7a4cc203539  returns_panel_v2_2026-07-21.parquet
    4241c7a4cc203539  returns_panel_v2_2026-07-22.parquet
    f4b74efb1c3747a0  returns_

In [26]:
# =============================================================================
# V3 VERIFICATION — did the market-closed-date fix actually work?   READ-ONLY
#
# The fix was proposed with a FALSIFIABLE PREDICTION, recorded before the
# rebuild. This cell tests it. A fix that improves coverage but breaks something
# else is not a fix.
# =============================================================================
import numpy as np

p = pd.read_parquet(resolve('feature_panel_with_pit_v2'))
p['Date'] = pd.to_datetime(p['Date'])

print("=" * 78); print("V3 VERIFICATION"); print("=" * 78)

REMOVED = [pd.Timestamp(d) for d in
           ['2014-04-24','2014-10-15','2026-01-15','2026-05-01','2026-05-28','2026-06-26']]

print("\
[A] STRUCTURE")
print(f"  rows           : {len(p):,}")
print(f"  distinct dates : {p['Date'].nunique():,}")
print(f"  tickers        : {p['ticker'].nunique():,}")
print(f"  removed dates absent : {not p['Date'].isin(REMOVED).any()}")
if 'date_status' in p.columns:
    print(f"  date_status    : {dict(p['date_status'].value_counts())}")

print("\
[B] D+1 RECOVERY — the day after a deleted holiday must now have a return")
# Before the fix, 2026-05-29 had NO daily_return for any ticker: its prior row
# was the fabricated 2026-05-28 holiday, whose close was NaN. After deletion,
# shift(1) lands on 2026-05-27 and the return computes normally.
for d in ['2026-05-29','2026-06-29','2026-01-16','2026-05-04']:
    sub = p[p['Date'] == pd.Timestamp(d)]
    if len(sub):
        n_ret = int(sub['daily_return'].notna().sum())
        pct = 100 * n_ret / len(sub)
        flag = "  <-- RECOVERED" if pct > 50 else "  <-- STILL BROKEN"
        print(f"  {d}: {n_ret:,}/{len(sub):,} valid returns ({pct:.1f}%){flag}")

print("\
[C] DAILY-RETURN COVERAGE BY YEAR")
p['_yr'] = p['Date'].dt.year
cov = p.groupby('_yr').agg(
    rows=('ticker','size'),
    daily_ret_pct=('daily_return', lambda s: 100*s.notna().mean()),
    vol120_pct=('realized_vol_120', lambda s: 100*s.notna().mean()))
print(cov.to_string(float_format=lambda x: f"{x:,.2f}"))

print("\
[D] THE PREDICTION")
print("  Recorded BEFORE the rebuild. 2014/2015/2025/2026 daily-return coverage")
print("  should return to the level of a normal year (~2023). If any stays")
print("  depressed, the six dates were NOT the whole story.")
base = cov.loc[2023, 'daily_ret_pct'] if 2023 in cov.index else float('nan')
print(f"\
  2023 baseline: {base:.2f}%")
for yr in [2014, 2015, 2024, 2025, 2026]:
    if yr in cov.index:
        v = cov.loc[yr, 'daily_ret_pct']
        verdict = "OK" if v >= 0.85 * base else "STILL DEPRESSED"
        print(f"  {yr}: {v:6.2f}%   [{verdict}]")

print("\
[E] NEXT STEP")
print("  Re-run the Phase A validation notebook against this panel. Coverage")
print("  here is necessary but not sufficient -- the momentum-eligible universe")
print("  (flag==1 AND full 232-row window) is the number that actually matters,")
print("  and only Phase A measures it. Expect ~450-490 in 2025/2026, vs 126/22")
print("  before the fix.")
print("=" * 78)


V3 VERIFICATION
[A] STRUCTURE
  rows           : 3,328,132
  distinct dates : 4,237
  tickers        : 1,069
  removed dates absent : True
  date_status    : {'normal': np.int64(3327217), 'vendor_failure': np.int64(915)}
[B] D+1 RECOVERY — the day after a deleted holiday must now have a return
  2026-05-29: 906/931 valid returns (97.3%)  <-- RECOVERED
  2026-06-29: 906/931 valid returns (97.3%)  <-- RECOVERED
  2026-01-16: 906/932 valid returns (97.2%)  <-- RECOVERED
  2026-05-04: 906/932 valid returns (97.2%)  <-- RECOVERED
[C] DAILY-RETURN COVERAGE BY YEAR
        rows  daily_ret_pct  vol120_pct
_yr                                    
2010  178551          97.46       48.28
2011  182428          98.27       89.91
2012  186033          98.14       89.96
2013  189456          97.67       91.97
2014  185025          98.36       91.60
2015  189485          97.94       91.98
2016  193184          97.95       93.14
2017  198979          98.07       94.65
2018  202672          98.36       9

In [27]:
p2025 = p[(p['Date'] >= '2024-09-01') & (p['Date'] <= '2025-12-31')]
by_date = p2025.groupby('Date')['realized_vol_120'].apply(lambda s: 100*s.notna().mean())
print("vol120 coverage around the vendor-failure window:")
print(by_date.loc['2025-02-25':'2025-04-05'].round(1).to_string())
print("\nMonthly:")
print(p2025.groupby(p2025['Date'].dt.to_period('M'))['realized_vol_120']
      .apply(lambda s: round(100*s.notna().mean(),1)).to_string())

vol120 coverage around the vendor-failure window:
Date
2025-02-25    93.0
2025-02-27    93.0
2025-02-28    93.1
2025-03-03    93.1
2025-03-04    93.1
2025-03-05    93.1
2025-03-06    93.1
2025-03-07    93.2
2025-03-10    93.2
2025-03-11    93.2
2025-03-12    93.2
2025-03-13    93.2
2025-03-17    93.2
2025-03-18     1.7
2025-03-19     1.7
2025-03-20     1.7
2025-03-21     1.7
2025-03-24     1.6
2025-03-25     1.6
2025-03-26     1.6
2025-03-27     1.6
2025-03-28     1.6
2025-04-01     1.6
2025-04-02     1.6
2025-04-03     1.5
2025-04-04     1.5

Monthly:
Date
2024-09    93.0
2024-10    93.0
2024-11    92.4
2024-12    92.3
2025-01    92.7
2025-02    92.9
2025-03    49.9
2025-04     1.5
2025-05     1.5
2025-06     1.5
2025-07     1.5
2025-08     1.5
2025-09    56.1
2025-10    93.6
2025-11    93.2
2025-12    93.3
Freq: M
